# ThyroFuse — Round 3 (editor's focused major revision): every required analysis, one notebook

Run once on the analysis PC; it reuses your Round 2 outputs and never overwrites them. Results go to `D:\Thyroid-BHI-26\revision_round3\`.

| Stage | What it produces | Editor points |
|---|---|---|
| Q | Masking QA: detection vs fallback rate and reasons, detected-box sizes, random visual audit, expert-review form | 6, 32, 35 |
| M | Masking control for DenseNet-121, ResNet-50, Swin-Tiny, ConvNeXt-Small (EfficientNet-B3 exists) — **GPU, ~12–14 h, resumable** | 5 |
| MA | Masking: pooled AUROC per condition and **paired ΔAUROC with case-clustered 95% CI** for every backbone and the ensemble | 3, 4, 5, 30, 47 |
| SD | Split-design: **paired ΔAUROC (image-level − case-grouped)** with case-clustered CI | 13, 28, 29 |
| MD | **Metadata-only shortcut baseline** (no CNN) on the frozen split and grouped CV; class-wise acquisition metadata | 7, 8, 41 |
| ND | Near-duplicate adjudication (SSIM, correlation) of every pair crossing frozen partitions; sensitivity analysis | 9 |
| HX | Systematically selected acquisition-context examples | 42 |
| GC | Grad-CAM with raw maps on one shared scale, perceptually uniform colour map, labelled boxes — GPU, ~5 min | 33, 34, 35 |
| FIG | All main and supplementary figures, vector PDF/SVG + PNG, one colour system | 21–55 |
| TAB | Main tables 1–4, supplementary tables, reproducibility index | 12, 20, 43–50 |
| Z | Checkpoint SHA-256, environment, Zenodo checkpoint archive, **one results zip to upload** | 18, 20 |

**How to run.** Check `TORCH_PY` and the paths in CONFIG, then Kernel → Restart & Run All. Stage M is long; keep the PC awake. Everything is resumable. If you need results sooner, set `MASK_MODELS = ["densenet121"]` — the manuscript would then restrict masking claims to EfficientNet-B3 and DenseNet-121 (no ensemble). At the end, upload `revision_round3\REVISION_ROUND3_UPLOAD.zip`.

**Expert review (editor point 6, optional but recommended).** Stage Q writes `masking_QA_expert_review_sheet.png` and `masking_QA_expert_review_form.csv` (40 images). If a radiologist or sonographer can mark each box as adequate / partial / inadequate coverage of the sonographic sector, save the form and re-run stages TAB and Z.

## CONFIG

In [ ]:
from pathlib import Path
ROOT            = Path(r"D:\Thyroid-BHI-26")
TORCH_PY        = Path(r"C:\Users\admin\miniconda3\envs\nnunet\python.exe")
FROZEN_MANIFEST = ROOT / "revision_final_frozen_70_15_15" / "04_frozen_manifest.csv"
FROZEN_SHA256   = "83e656eff33e272020d85f8faf4004285a67768d3ef79e040c59ee13ea1575f5"
EXCLUDED_CSV    = ROOT / "revision_final_frozen_70_15_15" / "01_excluded_cross_label_conflicts.csv"
SCRIPT_03B      = ROOT / "Rivision scripts" / "03b_group_stratified_5fold_cv.py"
EXP1B_DIR       = ROOT / "revision_exp1b"
CV_DIR          = ROOT / "revision_exp2b_group_cv"
MAY_SPLIT       = ROOT / "data" / "metadata_unpaired_leakage_safe.csv"
PRIOR_PAPER_CSV = Path(r"C:\Thyroid\metadata_with_holdout.csv")
ROUND2          = ROOT / "revision_round2"          # read only
ROUND3          = ROOT / "revision_round3"          # all new outputs

MODELS      = ["convnext_small", "efficientnet_b3", "swin_tiny", "resnet50", "densenet121"]
MASK_MODELS = ["densenet121", "resnet50", "swin_tiny", "convnext_small"]   # priority order; EfficientNet-B3 already done
DISPLAY     = {"convnext_small": "ConvNeXt-Small", "efficientnet_b3": "EfficientNet-B3", "swin_tiny": "Swin-Tiny",
               "resnet50": "ResNet-50", "densenet121": "DenseNet-121", "ensemble": "Ensemble (mean of 5)"}
BATCH_SIZE  = 16
NUM_WORKERS = 4
N_BOOT      = 2000
SEED        = 2026
ZIP_CHECKPOINTS_FOR_ZENODO = True

RUN = dict(PRE=True, Q=True, M=True, MA=True, SD=True, MD=True, ND=True, HX=True, GC=True, FIG=True, TAB=True, Z=True)

## Setup and helpers (from Round 2, retargeted to the Round 3 folder)

In [ ]:
import os, sys, re, io, json, time, shutil, hashlib, zipfile, datetime, subprocess, contextlib, traceback, warnings, itertools
import xml.etree.ElementTree as ET
from collections import Counter
import numpy as np
import pandas as pd

def _ensure(pkgs):
    missing = []
    for mod, pipname in pkgs:
        try:
            __import__(mod)
        except ImportError:
            missing.append(pipname)
    if missing:
        print("installing into this kernel:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
_ensure([("sklearn", "scikit-learn"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("PIL", "pillow")])

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
from PIL import Image, ImageFile
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

for sub in ["logs", "tables", "figures", "predictions", "scripts", "gradcam", "release_additions"]:
    (ROUND3 / sub).mkdir(parents=True, exist_ok=True)
RESULTS_MD = ROUND3 / "RESULTS.md"
if not RESULTS_MD.exists():
    RESULTS_MD.write_text("# Round-3 revision results\n", encoding="utf-8")
STATUS = {}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
                     "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False, "font.family": "DejaVu Sans"})

class _Tee(io.TextIOBase):
    def __init__(self, *s): self.s = s
    def write(self, x):
        for st in self.s: st.write(x)
        return len(x)
    def flush(self):
        for st in self.s: st.flush()

@contextlib.contextmanager
def stage(key, title):
    log = ROUND3 / "logs" / f"stage_{key}.log"
    with open(log, "a", encoding="utf-8") as fh, contextlib.redirect_stdout(_Tee(sys.stdout, fh)):
        t0 = time.time()
        print("\n" + "=" * 100 + f"\nSTAGE {key}: {title}   [{datetime.datetime.now():%Y-%m-%d %H:%M}]\n" + "=" * 100)
        if not RUN.get(key, True):
            print("skipped (RUN flag is False)"); STATUS[key] = "skipped"
            yield False
            return
        try:
            yield True
            STATUS[key] = f"done in {(time.time() - t0) / 60:.1f} min"
        except Exception:
            print("!!! STAGE FAILED — later stages continue. Traceback:\n" + traceback.format_exc())
            STATUS[key] = "FAILED (see log)"
        print(f"--- stage {key}: {STATUS.get(key)}")

def run_cmd(cmd, log_name, env_extra=None):
    """Run a command, streaming output live and to logs/<log_name>. Raises on non-zero exit."""
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    if env_extra: env.update(env_extra)
    log = ROUND3 / "logs" / log_name
    print("$ " + " ".join(str(c) for c in cmd) + (f"   [env {env_extra}]" if env_extra else ""))
    with open(log, "a", encoding="utf-8") as fh:
        fh.write(f"\n### {datetime.datetime.now():%Y-%m-%d %H:%M}  " + " ".join(str(c) for c in cmd) + "\n")
        p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace", env=env)
        last = time.time()
        for line in p.stdout:
            fh.write(line); fh.flush()
            if ("epoch" in line and time.time() - last < 20) and "CV SUMMARY" not in line:
                continue  # keep the notebook readable; the full log has every line
            print(line.rstrip()); last = time.time()
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed (exit {rc}); see {log}")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def parts(p):
    from pathlib import PureWindowsPath, PurePosixPath
    s = str(p); return PureWindowsPath(s).parts if "\\" in s else PurePosixPath(s).parts

def case_key_from_path(p):
    q = [x.lower() for x in parts(p)]
    folder = " ".join(q[:-1])
    lab = "PTC" if ("papillary" in folder or "carcinoma" in folder) else ("BEN" if "benign" in folder else None)
    m = re.match(r"^(\d+)", Path(q[-1]).stem)
    return f"{lab}_{int(m.group(1)):04d}" if (lab and m) else None

def img_id(p):
    return "/".join(x.lower() for x in parts(p)[-2:])

def save_table(df, name, title, note=None, show=True):
    df.to_csv(ROUND3 / "tables" / f"{name}.csv", index=False)
    with open(RESULTS_MD, "a", encoding="utf-8") as f:
        f.write(f"\n\n## {name}: {title}\n")
        if note: f.write(f"\n{note}\n")
        f.write("\n" + df.to_markdown(index=False) + "\n" if hasattr(df, "to_markdown") and _has_tabulate() else "\n```\n" + df.to_string(index=False) + "\n```\n")
    if show:
        print(f"\n[{name}] {title}"); print(df.to_string(index=False))

def _has_tabulate():
    try:
        import tabulate  # noqa
        return True
    except ImportError:
        return False

def note_md(text):
    with open(RESULTS_MD, "a", encoding="utf-8") as f: f.write("\n" + text + "\n")
    print(text)

In [ ]:
# ------------------------------- statistics -------------------------------
def youden(y, p):
    """Identical to 03b.youden."""
    fpr, tpr, thr = roc_curve(np.asarray(y, int), np.asarray(p, float))
    finite = np.where(np.isfinite(thr))[0]
    if len(finite) == 0: return 0.5
    return float(thr[finite[np.argmax(tpr[finite] - fpr[finite])]])

def ece15(y, p, bins=15):
    """Expected calibration error: 15 equal-width bins on [0,1], weighted by bin count; bin i = (i/15, (i+1)/15]."""
    y = np.asarray(y, float); p = np.asarray(p, float)
    idx = np.clip(np.ceil(p * bins).astype(int) - 1, 0, bins - 1)
    return float(sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any()))

def op_metrics(y, p, thr):
    y = np.asarray(y, int); pr = (np.asarray(p, float) >= thr).astype(int)
    tp = int(((pr == 1) & (y == 1)).sum()); tn = int(((pr == 0) & (y == 0)).sum())
    fp = int(((pr == 1) & (y == 0)).sum()); fn = int(((pr == 0) & (y == 1)).sum())
    d = lambda a, b: a / b if b else np.nan
    return dict(threshold=float(thr), tn=tn, fp=fp, fn=fn, tp=tp, accuracy=d(tp + tn, len(y)), sensitivity=d(tp, tp + fn),
                specificity=d(tn, tn + fp), ppv=d(tp, tp + fp), npv=d(tn, tn + fn), f1=d(2 * tp, 2 * tp + fp + fn))

def full_metrics(y, p, thr=None):
    y = np.asarray(y, int); p = np.asarray(p, float); two = len(np.unique(y)) == 2
    out = dict(n=len(y), n_pos=int(y.sum()), n_neg=int((1 - y).sum()), prevalence=float(y.mean()) if len(y) else np.nan,
               auroc=roc_auc_score(y, p) if two else np.nan, auprc=average_precision_score(y, p) if two else np.nan,
               brier=float(np.mean((p - y) ** 2)), ece15=ece15(y, p))
    if thr is not None: out.update(op_metrics(y, p, thr))
    return out

def boot_ci(y, p, groups, thr=None, n_boot=None, seed=SEED):
    """Percentile 95% CIs from resampling whole groups (cases/components) with replacement."""
    n_boot = n_boot or N_BOOT
    y = np.asarray(y, int); p = np.asarray(p, float); g = np.asarray(groups).astype(str)
    uniq, inv = np.unique(g, return_inverse=True)
    members = [np.where(inv == k)[0] for k in range(len(uniq))]
    rng = np.random.default_rng(seed); rec = {k: [] for k in ["auroc", "auprc", "brier", "sensitivity", "specificity", "accuracy"]}
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), len(uniq))])
        yy, pp = y[idx], p[idx]
        if len(np.unique(yy)) < 2: continue
        rec["auroc"].append(roc_auc_score(yy, pp)); rec["auprc"].append(average_precision_score(yy, pp))
        rec["brier"].append(np.mean((pp - yy) ** 2))
        if thr is not None:
            o = op_metrics(yy, pp, thr)
            for k in ("sensitivity", "specificity", "accuracy"): rec[k].append(o[k])
    return {k: (float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))) for k, v in rec.items() if v}

def ci_str(v, ci):
    return "NA" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f} ({ci[0]:.3f}–{ci[1]:.3f})"

def to_logit(p):
    p = np.clip(np.asarray(p, float), 1e-7, 1 - 1e-7); return np.log(p / (1 - p))

def fit_temperature(y, p):
    z = to_logit(p); y = np.asarray(y, float)
    nll = lambda T: -np.mean(y * np.log(1 / (1 + np.exp(-z / T)) + 1e-12) + (1 - y) * np.log(1 - 1 / (1 + np.exp(-z / T)) + 1e-12))
    return float(minimize_scalar(nll, bounds=(0.05, 100.0), method="bounded").x)

def apply_temperature(p, T):
    return 1 / (1 + np.exp(-to_logit(p) / T))

def reliability(y, p, bins=10):
    y = np.asarray(y, float); p = np.asarray(p, float); edges = np.linspace(0, 1, bins + 1); rows = []
    for i in range(bins):
        m = (p > edges[i]) & (p <= edges[i + 1]) if i else (p >= 0) & (p <= edges[1])
        if m.any(): rows.append(dict(bin=i, mean_pred=p[m].mean(), frac_pos=y[m].mean(), n=int(m.sum())))
    return pd.DataFrame(rows)

def near_binary_fraction(path):
    a = np.asarray(Image.open(path).convert("L"), dtype=np.uint8)
    return float(((a <= 10) | (a >= 245)).mean())
print("helpers ready | python", sys.version.split()[0])

## Verified Round 2 code: sector detector, pHash, 03b patches, inference script

In [ ]:
# Shared code (verified): sector detector, pHash identical to imagehash, patches to 03b, GPU inference script
SECTOR_SRC = r'''
# ---- sector box detector (shared by the notebook and the shortcut-control training script) ----
import hashlib as _hl
import numpy as _np
from PIL import Image as _Image
try:
    from scipy import ndimage as _ndi
except Exception:  # scipy is a scikit-learn dependency, so this should not happen
    _ndi = None

_SECTOR_CACHE = {}
SECTOR_FALLBACK = (0.12, 0.12, 0.88, 0.90)

def sector_box(img):
    """Bounding box (x0, y0, x1, y1 as fractions) of the largest textured region = the B-mode sector.
    Thin burned-in text and markers are removed by a 7x7 morphological opening.
    Falls back to a fixed central box when the detected region is implausibly small or large."""
    g = _np.asarray(img.convert("L"), dtype=_np.float32)
    h, w = g.shape
    key = (w, h, _hl.md5(_np.asarray(img.convert("L").resize((32, 32))).tobytes()).hexdigest())
    if key in _SECTOR_CACHE:
        return _SECTOR_CACHE[key]
    result = (SECTOR_FALLBACK, "fallback")
    if _ndi is not None and h > 20 and w > 20:
        m = _ndi.uniform_filter(g, 9)
        s = _np.sqrt(_np.clip(_ndi.uniform_filter(g * g, 9) - m * m, 0, None))
        tex = (s > 6.0) & (g > 12.0)
        tex = _ndi.binary_opening(tex, structure=_np.ones((7, 7), bool))
        lab, n = _ndi.label(tex)
        if n > 0:
            sizes = _ndi.sum(tex, lab, range(1, n + 1))
            k = int(_np.argmax(sizes)) + 1
            ys, xs = _np.where(lab == k)
            box = (xs.min() / w, ys.min() / h, (xs.max() + 1) / w, (ys.max() + 1) / h)
            area = (box[2] - box[0]) * (box[3] - box[1])
            if 0.25 <= area <= 0.97:
                result = (tuple(float(v) for v in box), "detected")
    _SECTOR_CACHE[key] = result
    return result

class SectorMask:
    """mode='surround_only' blacks out the sector; mode='center_only' blacks out everything outside it."""
    def __init__(self, mode):
        if mode not in ("surround_only", "center_only"):
            raise ValueError(mode)
        self.mode = mode
    def __call__(self, img):
        (x0, y0, x1, y1), _ = sector_box(img)
        arr = _np.array(img.convert("RGB"))
        h, w = arr.shape[:2]
        X0, Y0 = int(x0 * w), int(y0 * h)
        X1, Y1 = int(_np.ceil(x1 * w)), int(_np.ceil(y1 * h))
        if self.mode == "surround_only":
            out = arr.copy(); out[Y0:Y1, X0:X1] = 0
        else:
            out = _np.zeros_like(arr); out[Y0:Y1, X0:X1] = arr[Y0:Y1, X0:X1]
        return _Image.fromarray(out)
'''

PHASH_SRC = r'''
# ---- pHash identical to imagehash.phash (hash_size=8, highfreq_factor=4) ----
import numpy as _np2
from PIL import Image as _Image2
from scipy.fftpack import dct as _dct

def phash_hex(img):
    try:
        lanczos = _Image2.Resampling.LANCZOS
    except AttributeError:
        lanczos = _Image2.LANCZOS
    px = _np2.asarray(img.convert("L").resize((32, 32), lanczos), dtype=_np2.float64)
    d = _dct(_dct(px, axis=0), axis=1)[:8, :8]
    bits = (d > _np2.median(d)).flatten()
    return "{:016x}".format(int("".join("1" if b else "0" for b in bits), 2))

def hex_to_u64(h):
    return _np2.uint64(int(str(h), 16))

def hamming_matrix(a_hex, b_hex):
    a = _np2.array([int(str(x), 16) for x in a_hex], dtype=_np2.uint64)
    b = _np2.array([int(str(x), 16) for x in b_hex], dtype=_np2.uint64)
    x = _np2.bitwise_xor(a[:, None], b[None, :])
    return _np2.unpackbits(x.view(_np2.uint8).reshape(x.shape + (8,)), axis=-1).sum(-1).astype(_np2.int16)
'''

PATCH_SRC = r'''
def patch_03b_imagelevel(src):
    """Only change: the CV grouping key becomes the individual image instead of the duplicate-connected case component."""
    old = '+ "::component" + manifest["component_id"].astype(str)'
    new = '+ "::image" + manifest["image_path"].astype(str)  # ROUND2 PATCH: image-level folds (leakage demonstration)'
    if src.count(old) != 1:
        raise RuntimeError("03b grouping line not found exactly once; script version differs from the audited one")
    out = src.replace(old, new)
    out = out.replace('print("Grouping unit   : duplicate-connected component_id")',
                      'print("Grouping unit   : INDIVIDUAL IMAGE (round-2 leakage demonstration)")')
    return out

def patch_03b_shortcut(src, sector_src):
    """Only change: a SectorMask transform is prepended to the train and eval pipelines (mode from THYRO_MASK_MODE)."""
    anchor = "def transforms_for_run():"
    if src.count(anchor) != 1:
        raise RuntimeError("transforms_for_run() not found exactly once")
    old = "transforms.Compose([\n        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),"
    new = "transforms.Compose([\n        SectorMask(MASK_MODE),  # ROUND2 PATCH\n        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),"
    if src.count(old) != 2:
        raise RuntimeError(f"expected 2 Compose/Resize blocks, found {src.count(old)}")
    header = sector_src + '\nMASK_MODE = os.environ.get("THYRO_MASK_MODE", "")\n' \
             'if MASK_MODE not in ("surround_only", "center_only"):\n' \
             '    raise RuntimeError("Set THYRO_MASK_MODE to surround_only or center_only")\n\n\n'
    return src.replace(old, new).replace(anchor, header + anchor)
'''

GPU_TASKS_SRC = r'''
"""Round-2 GPU tasks (inference only; no training).
Imports the audited 03b script, so datasets, transforms, prediction and fold construction
are identical to the cross-validation runs that produced the checkpoints."""
import argparse, importlib.util, json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image


def load_module(path, name="exp2b"):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def tload(path, device):
    try:
        return torch.load(str(path), map_location=device, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=device)


def device_of():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model(m, key, ckpt, device):
    model = timm.create_model(m.MODELS[key], pretrained=False, num_classes=1)
    state = tload(ckpt, device)
    sd = state["model_state_dict"] if isinstance(state, dict) and "model_state_dict" in state else state
    model.load_state_dict(sd)
    return model.to(device).eval()


def free(model):
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def with_labels(df):
    df = df.reset_index(drop=True).copy()
    if "label" not in df.columns:
        df["label"] = -1
    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(-1)
    return df


def cmd_cv_innerval(a):
    m = load_module(a.script03b)
    dev = device_of()
    manifest = pd.read_csv(a.manifest, low_memory=False)
    manifest["_component_uid"] = (manifest["modality"].astype(str).str.lower()
                                  + "::component" + manifest["component_id"].astype(str))
    devset = manifest[manifest["split"].isin(["train", "val"])].copy()
    mdev = devset[devset["modality"].astype(str).str.lower().eq(a.modality)].copy().reset_index(drop=True)
    outer = m.make_outer_splits(mdev, m.SEED)
    _, eval_tf = m.transforms_for_run()
    ref = None
    if a.fold_assignment and Path(a.fold_assignment).exists():
        ref = pd.read_csv(a.fold_assignment)
        ref = ref[ref["modality"].astype(str).str.lower().eq(a.modality)]
    for fold, (tr_idx, ho_idx) in enumerate(outer, 1):
        fold_seed = m.SEED + fold * 100
        outer_train = mdev.iloc[tr_idx].reset_index(drop=True)
        hold = mdev.iloc[ho_idx]
        if ref is not None and len(ref):
            r = set(ref.loc[ref["outer_fold"].eq(fold), "image_path"])
            if r != set(hold["image_path"]):
                raise RuntimeError(f"{a.modality} fold {fold}: replicated holdout differs from fold_assignment.csv")
        _, iv_idx = m.make_inner_split(outer_train, fold_seed + 1)
        inner_val = outer_train.iloc[iv_idx].reset_index(drop=True)
        loader = m.make_loader(inner_val, eval_tf, a.batch_size, 0)
        for key in a.models:
            dst = Path(a.out) / a.modality / key / f"fold_{fold}_innerval_predictions.csv"
            if dst.exists():
                print("[skip]", dst, flush=True)
                continue
            ckpt = Path(a.cv_dir) / a.modality / key / f"fold_{fold}" / "best_model.pt"
            if not ckpt.exists():
                raise FileNotFoundError(ckpt)
            model = build_model(m, key, ckpt, dev)
            pred = m.predict(model, loader, inner_val, dev)
            thr = float(m.youden(pred["label"], pred["prob_malignant"]))
            out = pred[["image_path", "label", "source_group", "component_id", "prob_malignant"]].copy()
            out["outer_fold"] = fold
            out["model_key"] = key
            out["recomputed_threshold"] = thr
            dst.parent.mkdir(parents=True, exist_ok=True)
            out.to_csv(dst, index=False)
            print(f"{a.modality} fold {fold} {key}: inner-val n={len(out)} threshold={thr:.6f}", flush=True)
            free(model)


def cmd_check_folds(a):
    """Exit 0 if 03b in THIS environment reproduces the saved fold assignment exactly (both modalities), else exit 3."""
    import sklearn
    m = load_module(a.script03b)
    manifest = pd.read_csv(a.manifest, low_memory=False)
    manifest["_component_uid"] = (manifest["modality"].astype(str).str.lower()
                                  + "::component" + manifest["component_id"].astype(str))
    devset = manifest[manifest["split"].isin(["train", "val"])]
    ref = pd.read_csv(a.fold_assignment)
    report = {}
    for mo in ["ultrasound", "cytology"]:
        r = ref[ref["modality"].astype(str).str.lower().eq(mo)]
        if r.empty:
            continue
        mdev = devset[devset["modality"].astype(str).str.lower().eq(mo)].copy().reset_index(drop=True)
        outer = m.make_outer_splits(mdev, m.SEED)
        report[mo] = all(set(r.loc[r["outer_fold"].eq(f), "image_path"]) == set(mdev.iloc[h]["image_path"])
                         for f, (_, h) in enumerate(outer, 1))
    ok = bool(report) and all(report.values())
    print(json.dumps({"sklearn": sklearn.__version__, "folds_reproduced": report, "ok": ok}), flush=True)
    sys.exit(0 if ok else 3)


def cmd_predict(a):
    m = load_module(a.script03b)
    dev = device_of()
    df = with_labels(pd.read_csv(a.manifest))
    _, eval_tf = m.transforms_for_run()
    loader = m.make_loader(df, eval_tf, a.batch_size, 0)
    outdir = Path(a.out)
    outdir.mkdir(parents=True, exist_ok=True)
    for key in a.models:
        dst = outdir / f"{key}_predictions.csv"
        if dst.exists():
            print("[skip]", dst, flush=True)
            continue
        ckpt = Path(a.exp1b_dir) / a.modality / key / "best_model.pt"
        if not ckpt.exists():
            raise FileNotFoundError(ckpt)
        t0 = time.time()
        model = build_model(m, key, ckpt, dev)
        pred = m.predict(model, loader, df, dev)
        out = df.copy()
        out["prob_malignant"] = pred["prob_malignant"].values
        out["model_key"] = key
        out.to_csv(dst, index=False)
        print(f"{Path(a.manifest).name} {key}: n={len(out)} in {time.time() - t0:.0f}s", flush=True)
        free(model)


def cmd_embed(a):
    m = load_module(a.script03b)
    dev = device_of()
    df = with_labels(pd.read_csv(a.manifest))
    outdir = Path(a.out)
    outdir.mkdir(parents=True, exist_ok=True)
    ckpt = Path(a.exp1b_dir) / a.modality / a.model / "best_model.pt"
    model = build_model(m, a.model, ckpt, dev)
    model.reset_classifier(0)
    _, eval_tf = m.transforms_for_run()
    loader = m.make_loader(df, eval_tf, a.batch_size, 0)
    feats = None
    with torch.no_grad():
        for images, _, idx in loader:
            f = model(images.to(dev)).float().cpu().numpy()
            if feats is None:
                feats = np.zeros((len(df), f.shape[1]), np.float32)
            feats[idx.numpy()] = f
    np.save(outdir / "embeddings.npy", feats)
    df.to_csv(outdir / "embeddings_index.csv", index=False)
    print("embeddings:", feats.shape, flush=True)


def cmd_gradcam(a):
    m = load_module(a.script03b)
    dev = device_of()
    df = pd.read_csv(a.manifest).reset_index(drop=True)
    outdir = Path(a.out)
    outdir.mkdir(parents=True, exist_ok=True)
    ckpt = Path(a.exp1b_dir) / a.modality / a.model / "best_model.pt"
    model = build_model(m, a.model, ckpt, dev)
    modules = dict(model.named_modules())
    if a.layer not in modules:
        raise KeyError(f"layer {a.layer} not in model; candidates end with: {list(modules)[-8:]}")
    store = {}

    def hook(mod, inp, out):
        store["act"] = out
        out.register_hook(lambda g: store.__setitem__("grad", g))

    handle = modules[a.layer].register_forward_hook(hook)
    _, eval_tf = m.transforms_for_run()
    S = 224
    cams = np.zeros((len(df), S, S), np.float16)
    rows = []
    for i, r in df.iterrows():
        rec = {"image_path": r["image_path"]}
        try:
            with Image.open(r["image_path"]) as im:
                x = eval_tf(im.convert("RGB")).unsqueeze(0).to(dev)
            model.zero_grad(set_to_none=True)
            logit = model(x).reshape(-1)[0]
            logit.backward()
            A, G = store["act"].detach(), store["grad"].detach()
            cam = torch.relu((G.mean(dim=(2, 3), keepdim=True) * A).sum(1, keepdim=True))
            cam = torch.nn.functional.interpolate(cam.float(), size=(S, S), mode="bilinear", align_corners=False)[0, 0]
            cam = cam.cpu().numpy()
            X0, Y0 = int(r["box_x0"] * S), int(r["box_y0"] * S)
            X1, Y1 = int(np.ceil(r["box_x1"] * S)), int(np.ceil(r["box_y1"] * S))
            total = float(cam.sum())
            inside = float(cam[Y0:Y1, X0:X1].sum())
            py, px = np.unravel_index(int(cam.argmax()), cam.shape)
            rec.update(prob_malignant=float(torch.sigmoid(logit).item()),
                       cam_mass_outside_sector=(1 - inside / total) if total > 0 else np.nan,
                       sector_area_outside=1 - ((X1 - X0) * (Y1 - Y0)) / float(S * S),
                       peak_outside_sector=not (X0 <= px < X1 and Y0 <= py < Y1),
                       status="ok")
            cams[i] = (cam / cam.max() if cam.max() > 0 else cam).astype(np.float16)
        except Exception as e:
            rec.update(status=f"error: {e}")
        rows.append(rec)
    handle.remove()
    np.savez_compressed(outdir / "cams.npz", cams=cams)
    pd.DataFrame(rows).to_csv(outdir / "gradcam_per_image.csv", index=False)
    print("grad-cam images:", len(rows), "errors:", sum(1 for r in rows if r["status"] != "ok"), flush=True)


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("cv-innerval")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--cv-dir", required=True); p.add_argument("--fold-assignment", default="")
    p.add_argument("--modality", required=True); p.add_argument("--models", nargs="+", required=True)
    p.add_argument("--out", required=True); p.add_argument("--batch-size", type=int, default=16)
    p = sub.add_parser("check-folds")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--fold-assignment", required=True)
    p = sub.add_parser("predict")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--exp1b-dir", required=True); p.add_argument("--modality", default="ultrasound")
    p.add_argument("--models", nargs="+", required=True); p.add_argument("--out", required=True)
    p.add_argument("--batch-size", type=int, default=16)
    p = sub.add_parser("embed")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--exp1b-dir", required=True); p.add_argument("--modality", default="ultrasound")
    p.add_argument("--model", default="efficientnet_b3"); p.add_argument("--out", required=True)
    p.add_argument("--batch-size", type=int, default=16)
    p = sub.add_parser("gradcam")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--exp1b-dir", required=True); p.add_argument("--modality", default="ultrasound")
    p.add_argument("--model", default="efficientnet_b3"); p.add_argument("--layer", default="conv_head")
    p.add_argument("--out", required=True)
    a = ap.parse_args()
    {"cv-innerval": cmd_cv_innerval, "check-folds": cmd_check_folds, "predict": cmd_predict, "embed": cmd_embed, "gradcam": cmd_gradcam}[a.cmd](a)


if __name__ == "__main__":
    main()
'''


exec(SECTOR_SRC); exec(PHASH_SRC); exec(PATCH_SRC)
GPU_TASKS = ROUND3 / "scripts" / "r2_gpu_tasks.py"
GPU_TASKS.write_text(GPU_TASKS_SRC, encoding="utf-8")
PATCHED_IMAGELEVEL = ROUND3 / "scripts" / "03b_imagelevel_cv.py"
PATCHED_SHORTCUT = ROUND3 / "scripts" / "03b_shortcut_cv.py"
if SCRIPT_03B.exists():
    _src = SCRIPT_03B.read_text(encoding="utf-8")
    PATCHED_IMAGELEVEL.write_text(patch_03b_imagelevel(_src), encoding="utf-8")
    PATCHED_SHORTCUT.write_text(patch_03b_shortcut(_src, SECTOR_SRC), encoding="utf-8")
    print("wrote patched scripts:", PATCHED_IMAGELEVEL.name, PATCHED_SHORTCUT.name)
print("wrote", GPU_TASKS)

## Round 3 helpers

In [ ]:
# ---------------------------------------------------------------- general
def read_frozen():
    fm = pd.read_csv(FROZEN_MANIFEST, low_memory=False)
    fm["case_key"] = fm["image_path"].map(case_key_from_path)
    return fm

def parse_ci(s):
    m = re.match(r"\s*([0-9.]+)\s*\(([0-9.]+)\s*[–-]\s*([0-9.]+)\)", str(s))
    return (float(m.group(1)), float(m.group(2)), float(m.group(3))) if m else (np.nan, np.nan, np.nan)

def fmt3(v, lo, hi): return f"{v:.3f} ({lo:.3f} to {hi:.3f})"

def paired_boot(y, pa, pb, groups, n_boot=None, seed=SEED):
    """Paired case-clustered bootstrap. Returns AUROC of a and b, delta = b - a, percentile 95% CIs and a two-sided bootstrap p."""
    n_boot = n_boot or N_BOOT
    y = np.asarray(y, int); pa = np.asarray(pa, float); pb = np.asarray(pb, float); g = np.asarray(groups).astype(str)
    uniq, inv = np.unique(g, return_inverse=True); members = [np.where(inv == k)[0] for k in range(len(uniq))]
    rng = np.random.default_rng(seed); A, B, D = [], [], []
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), len(uniq))])
        if len(np.unique(y[idx])) < 2: continue
        a = roc_auc_score(y[idx], pa[idx]); b = roc_auc_score(y[idx], pb[idx]); A.append(a); B.append(b); D.append(b - a)
    A, B, D = map(np.asarray, (A, B, D))
    auc_a, auc_b = roc_auc_score(y, pa), roc_auc_score(y, pb)
    p = min(1.0, 2 * min((D <= 0).mean(), (D >= 0).mean())); p = max(p, 1.0 / len(D))
    pc = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return dict(auc_a=auc_a, auc_b=auc_b, delta=auc_b - auc_a, ci_a=pc(A), ci_b=pc(B), ci_delta=pc(D), p=p, n_boot=len(D))

def fold_leak_share(o):
    shares = []
    for f in sorted(o.outer_fold.unique()):
        hold = o[o.outer_fold == f]; train_groups = set(o[o.outer_fold != f].source_group)
        shares.append(hold.source_group.isin(train_groups).mean())
    return float(np.mean(shares))

# ---------------------------------------------------------------- instrumented sector detector (identical decisions to sector_box)
def sector_box_detail(img):
    g = np.asarray(img.convert("L"), dtype=np.float32); h, w = g.shape
    out = dict(width=w, height=h, status="fallback", reason="", box=SECTOR_FALLBACK, candidate_area=np.nan, n_components=0)
    if _ndi is None or h <= 20 or w <= 20:
        out["reason"] = "image too small"; return out
    m = _ndi.uniform_filter(g, 9); s = np.sqrt(np.clip(_ndi.uniform_filter(g * g, 9) - m * m, 0, None))
    tex = _ndi.binary_opening((s > 6.0) & (g > 12.0), structure=np.ones((7, 7), bool))
    lab, n = _ndi.label(tex); out["n_components"] = int(n)
    if n == 0:
        out["reason"] = "no textured region found"; return out
    sizes = _ndi.sum(tex, lab, range(1, n + 1)); k = int(np.argmax(sizes)) + 1
    ys, xs = np.where(lab == k)
    box = (xs.min() / w, ys.min() / h, (xs.max() + 1) / w, (ys.max() + 1) / h)
    area = (box[2] - box[0]) * (box[3] - box[1]); out["candidate_area"] = float(area)
    if area < 0.25: out["reason"] = "largest textured region < 25% of frame"
    elif area > 0.97: out["reason"] = "largest textured region > 97% of frame"
    else: out.update(status="detected", box=tuple(float(v) for v in box))
    return out

# ---------------------------------------------------------------- metadata features (no CNN, no anatomy-specific processing)
def meta_features(path):
    p = Path(path); im = Image.open(p); fmt = im.format or ""; mode = im.mode; w, h = im.size
    try: exif = int(len(im.getexif()) > 0)
    except Exception: exif = 0
    q = getattr(im, "quantization", None); jq = float(np.mean(q[0])) if q and 0 in q else np.nan
    rgb = im.convert("RGB"); rgb.thumbnail((512, 512)); a = np.asarray(rgb, dtype=np.float32)
    gray = a.mean(2); chroma = a.max(2) - a.min(2); H, W = gray.shape; sh, sw = max(1, int(.08 * H)), max(1, int(.08 * W))
    strips = {"top": (slice(0, sh), slice(None)), "bottom": (slice(H - sh, H), slice(None)), "left": (slice(None), slice(0, sw)), "right": (slice(None), slice(W - sw, W))}
    f = dict(width=w, height=h, aspect=w / h, n_bands=len(im.getbands()), pil_mode=mode, file_format=fmt,
             ext_uppercase=int(p.suffix != p.suffix.lower()), file_kb=p.stat().st_size / 1024, jpeg_q_mean=jq, exif_present=exif,
             gray_mean=gray.mean(), gray_sd=gray.std(), gray_p05=np.percentile(gray, 5), gray_p50=np.percentile(gray, 50), gray_p95=np.percentile(gray, 95),
             near_black=float((gray < 10).mean()), near_white=float((gray > 245).mean()), colored_frac=float((chroma > 30).mean()))
    for k, (r_, c_) in strips.items():
        f[f"{k}_mean"] = float(gray[r_, c_].mean()); f[f"{k}_near_white"] = float((gray[r_, c_] > 245).mean()); f[f"{k}_colored"] = float((chroma[r_, c_] > 30).mean())
    return f

FEATURE_SETS = {
    "A: frame geometry + global intensity": ["width", "height", "aspect", "n_bands", "gray_mean", "gray_sd", "gray_p05", "gray_p50", "gray_p95", "near_black", "near_white", "colored_frac"],
}
FEATURE_SETS["B: A + border strips (overlays)"] = FEATURE_SETS["A: frame geometry + global intensity"] + [f"{s}_{k}" for s in ["top", "bottom", "left", "right"] for k in ["mean", "near_white", "colored"]]
FEATURE_SETS["C: B + file encoding"] = FEATURE_SETS["B: A + border strips (overlays)"] + ["ext_uppercase", "file_kb", "jpeg_q_mean", "exif_present"]

# ---------------------------------------------------------------- image similarity for near-duplicate adjudication
def _gray256(path): return np.asarray(Image.open(path).convert("L").resize((256, 256)), dtype=np.float64)
def ssim256(a, b):
    from scipy.ndimage import uniform_filter as uf
    C1, C2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    ma, mb = uf(a, 7), uf(b, 7); va, vb = uf(a * a, 7) - ma ** 2, uf(b * b, 7) - mb ** 2; cab = uf(a * b, 7) - ma * mb
    return float((((2 * ma * mb + C1) * (2 * cab + C2)) / ((ma ** 2 + mb ** 2 + C1) * (va + vb + C2))).mean())

# ---------------------------------------------------------------- one restrained colour system (editor points 51-54)
COL = dict(cyto="#2f5d8a", us="#a0522d", ext="#6b6b8f", chance="#9a9a9a", ens="#111111", light="#d9d9d9")
MODEL_COL = {"convnext_small": "#4c72b0", "efficientnet_b3": "#55a868", "swin_tiny": "#8172b2", "resnet50": "#c44e52", "densenet121": "#937860", "ensemble": "#111111"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8, "legend.fontsize": 7,
                     "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.dpi": 300, "pdf.fonttype": 42, "svg.fonttype": "none"})
def save_fig(fig, name, vector=True):
    for ext in (["png", "pdf", "svg"] if vector else ["png", "pdf"]):
        fig.savefig(ROUND3 / "figures" / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print("figure:", name)
def panel(ax, letter): ax.text(-0.12, 1.06, letter, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")
print("round-3 helpers ready")

## Preflight

In [ ]:
with stage("PRE", "Preflight") as go:
  if go:
    checks = []
    def chk(name, ok, detail=""): checks.append(dict(check=name, ok="yes" if ok else "NO", detail=str(detail)))
    chk("frozen manifest SHA-256", sha256_file(FROZEN_MANIFEST) == FROZEN_SHA256)
    for k in MODELS: chk(f"grouped-CV OOF {k}", (CV_DIR / "ultrasound" / k / "oof_predictions.csv").exists())
    chk("EfficientNet-B3 masking (Round 2)", all((ROUND2 / "shortcut_cv" / m / "ultrasound" / "efficientnet_b3" / "oof_predictions.csv").exists() for m in ["surround_only", "center_only"]))
    chk("image-level split-design OOF (Round 2)", all((ROUND2 / "imagelevel_cv" / "cytology" / k / "oof_predictions.csv").exists() for k in ["convnext_small", "efficientnet_b3"]))
    chk("Round 2 tables", (ROUND2 / "tables" / "C1_cv_summary.csv").exists())
    chk("Round 2 Grad-CAM manifest", (ROUND2 / "manifests" / "gradcam_manifest.csv").exists())
    chk("fold assignments", all((ROUND2 / "manifests" / f"cv_fold_assignment_{m}.csv").exists() for m in ["ultrasound", "cytology"]))
    r = subprocess.run([str(TORCH_PY), "-c", "import torch,timm;print(torch.__version__,torch.cuda.is_available())"], capture_output=True, text=True)
    chk("TORCH_PY with CUDA", r.returncode == 0 and "True" in r.stdout, (r.stdout or r.stderr).strip()[-120:])
    todo = [(k, m) for k in MASK_MODELS for m in ["surround_only", "center_only"] if not (ROUND2 / "shortcut_cv" / m / "ultrasound" / k / "cv_summary.json").exists()]
    chk("masking runs still to do (Stage M)", True, f"{len(todo)}: {todo}")
    save_table(pd.DataFrame(checks), "PRE_checks", "Preflight", show=True)

## Stage Q — Masking QA (editor points 6, 32, 35)

In [ ]:
def draw_box(ax, im, det, lw=1.2):
    W, H = im.size; x0, y0, x1, y1 = det["box"]; fb = det["status"] != "detected"
    ax.imshow(im.convert("L"), cmap="gray"); ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(Rectangle((x0 * W, y0 * H), (x1 - x0) * W, (y1 - y0) * H, fill=False, lw=lw, ls="--" if fb else "-", ec="#f0a030" if fb else "#40c0ff"))

with stage("Q", "Masking QA") as go:
  if go:
    fm = read_frozen(); us = fm[fm.modality == "ultrasound"].reset_index(drop=True)
    rows, mismatch = [], 0
    for r in us.itertuples():
        im = Image.open(r.image_path); det = sector_box_detail(im); ref = sector_box(im)
        if ref[1] != det["status"] or max(abs(a - b) for a, b in zip(ref[0], det["box"])) > 1e-9: mismatch += 1
        rows.append(dict(image_path=r.image_path, case_key=r.case_key, label=int(r.label), split=r.split, width=det["width"], height=det["height"],
                         status=det["status"], fallback_reason=det["reason"], candidate_area=det["candidate_area"], n_components=det["n_components"],
                         box_x0=det["box"][0], box_y0=det["box"][1], box_x1=det["box"][2], box_y1=det["box"][3]))
    assert mismatch == 0, f"instrumented detector disagrees with the detector used for masking in {mismatch} images"
    Q = pd.DataFrame(rows); Q["box_area"] = (Q.box_x1 - Q.box_x0) * (Q.box_y1 - Q.box_y0)
    Q["box_width_frac"] = Q.box_x1 - Q.box_x0; Q["box_height_frac"] = Q.box_y1 - Q.box_y0
    Q.to_csv(ROUND3 / "tables" / "Q0_masking_qa_per_image.csv", index=False)
    n = len(Q); det_n = int((Q.status == "detected").sum())
    s1 = [dict(group="all analysed ultrasound images", images=n, detected=det_n, detected_pct=round(100 * det_n / n, 1), fallback=n - det_n, fallback_pct=round(100 * (n - det_n) / n, 1))]
    groups = [(("class", "benign" if l == 0 else "PTC"), g) for l, g in Q.groupby("label")] + [(("partition", sp), g) for sp, g in Q.groupby("split")]
    for (k, v), g in groups:
        d_ = int((g.status == "detected").sum()); s1.append(dict(group=f"{k}: {v}", images=len(g), detected=d_, detected_pct=round(100 * d_ / len(g), 1), fallback=len(g) - d_, fallback_pct=round(100 * (len(g) - d_) / len(g), 1)))
    save_table(pd.DataFrame(s1), "Q1_detection_rates", "Sonographic-sector detection: detected vs fallback, overall, by class and by partition")
    save_table(Q[Q.status != "detected"].groupby("fallback_reason").size().reset_index(name="images"), "Q2_fallback_reasons", "Reasons for falling back to the fixed central box")
    qd = Q[Q.status == "detected"]; qf = Q[Q.status != "detected"]
    desc = lambda s: f"{s.median():.2f} [{s.quantile(.25):.2f}–{s.quantile(.75):.2f}] (range {s.min():.2f}–{s.max():.2f})"
    save_table(pd.DataFrame([dict(boxes="detected", n=len(qd), area_fraction=desc(qd.box_area), width_fraction=desc(qd.box_width_frac), height_fraction=desc(qd.box_height_frac)),
                             dict(boxes="fallback: largest textured region found", n=int(qf.candidate_area.notna().sum()), area_fraction=desc(qf.candidate_area.dropna()) if qf.candidate_area.notna().any() else "NA", width_fraction="", height_fraction="")]),
               "Q3_box_sizes", "Detected-box size as a fraction of the frame: median [IQR] (range)")
    rng = np.random.default_rng(SEED)
    def sheet(df_, title, name, k=12, label_ids=None):
        df_ = df_.reset_index(drop=True)
        if len(df_) == 0: return
        cols = 6; rows_ = int(np.ceil(len(df_) / cols)); fig, axes = plt.subplots(rows_, cols, figsize=(9.5, 1.75 * rows_ + 0.4)); axes = np.atleast_2d(axes)
        for ax in axes.ravel(): ax.axis("off")
        for i, r in df_.iterrows():
            ax = axes.ravel()[i]; ax.axis("on"); im = Image.open(r.image_path)
            draw_box(ax, im, dict(box=(r.box_x0, r.box_y0, r.box_x1, r.box_y1), status=r.status))
            lab = (label_ids[i] + " · ") if label_ids else ""
            ax.set_title(f"{lab}{r.case_key} · {'detected' if r.status == 'detected' else 'fallback'}", fontsize=6)
        fig.suptitle(title, fontsize=8); fig.tight_layout(); save_fig(fig, name, vector=False)
    pick = lambda df_, k: df_.sample(min(k, len(df_)), random_state=SEED)
    sheet(pick(qd, 12), "Random sample of detected sonographic sectors (solid box)", "S_QA_detected_random")
    sheet(pick(qf, 12), "Random sample of fallback boxes (dashed): detection failed, fixed central box used", "S_QA_fallback_random")
    sheet(qd.nsmallest(min(6, len(qd)), "box_area"), "Detected boxes with the smallest areas (candidates for partial coverage)", "S_QA_smallest_detected")
    rev = pd.concat([pick(qd, 20), pick(qf, 20)]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    rev["review_id"] = [f"R{i + 1:02d}" for i in range(len(rev))]
    sheet(rev, "Expert review: does the box cover the sonographic sector? (record answers in the form)", "masking_QA_expert_review_sheet", label_ids=list(rev.review_id))
    form = rev[["review_id", "case_key", "status", "image_path"]].copy(); form["coverage_adequate_partial_inadequate"] = ""; form["reviewer_comment"] = ""
    fpath = ROUND3 / "masking_QA_expert_review_form.csv"
    if not fpath.exists(): form.to_csv(fpath, index=False)
    ex = {"detected (median-size box)": qd.iloc[(qd.box_area - qd.box_area.median()).abs().argsort().iloc[0]].image_path if len(qd) else None,
          "detected (smallest box)": qd.nsmallest(1, "box_area").iloc[0].image_path if len(qd) else None,
          "fallback central box": qf.sample(1, random_state=SEED).iloc[0].image_path if len(qf) else None}
    (ROUND3 / "tables" / "Q4_main_figure_examples.json").write_text(json.dumps(ex, indent=1))

## Stage M — Masking control for the remaining backbones (GPU, resumable) — editor point 5

In [ ]:
with stage("M", "Masking runs") as go:
  if go:
    patched = patch_03b_shortcut(SCRIPT_03B.read_text(encoding="utf-8"), SECTOR_SRC)
    PATCHED_SHORTCUT.write_text(patched, encoding="utf-8")
    r2p = ROUND2 / "scripts" / "03b_shortcut_cv.py"
    if r2p.exists():
        same = sha256_file(r2p) == sha256_file(PATCHED_SHORTCUT)
        note_md(f"- Masking script for new backbones is {'byte-identical to' if same else 'DIFFERENT FROM'} the Round 2 script used for EfficientNet-B3.")
        if not same: raise RuntimeError("masking script differs from Round 2 — stop and compare before training")
    for key in MASK_MODELS:
        for mode in ["surround_only", "center_only"]:
            out = ROUND2 / "shortcut_cv" / mode
            if (out / "ultrasound" / key / "cv_summary.json").exists():
                print(f"[done] {mode}/{key}"); continue
            run_cmd([TORCH_PY, PATCHED_SHORTCUT, "--models", key, "--modality", "ultrasound", "--out", out, "--batch-size", BATCH_SIZE,
                     "--num-workers", NUM_WORKERS], f"M_{mode}_{key}.log", env_extra={"THYRO_MASK_MODE": mode})

## Stage MA — Masking: absolute and paired ΔAUROC (editor points 3, 4, 5, 30, 47)

In [ ]:
COND = {"full image": None, "detected sector removed": "surround_only", "detected sector only": "center_only"}
def oof_path(key, mode):
    return (CV_DIR / "ultrasound" / key / "oof_predictions.csv") if mode is None else (ROUND2 / "shortcut_cv" / mode / "ultrasound" / key / "oof_predictions.csv")

with stage("MA", "Masking analysis") as go:
  if go:
    avail = [k for k in MODELS if all(oof_path(k, m).exists() for m in COND.values())]
    note_md(f"- Masking available for: {[DISPLAY[k] for k in avail]}; ensemble {'included' if len(avail) == len(MODELS) else 'not available (requires all five backbones)'}.")
    merged = {}
    for k in avail:
        base = None
        for cname, mode in COND.items():
            o = pd.read_csv(oof_path(k, mode), low_memory=False)[["image_path", "label", "component_id", "outer_fold", "prob_malignant"]].rename(
                columns={"prob_malignant": f"p_{cname}", "outer_fold": f"fold_{cname}", "label": f"y_{cname}"})
            base = o if base is None else base.merge(o.drop(columns=["component_id"]), on="image_path", how="inner", validate="one_to_one")
        assert len(base) > 0, f"{k}: no images in common across conditions"
        for c in COND:
            assert (base[f"y_{c}"] == base["y_full image"]).all() and (base[f"fold_{c}"] == base["fold_full image"]).all(), f"{k}: conditions disagree on labels or folds"
        merged[k] = base.rename(columns={"y_full image": "label", "fold_full image": "outer_fold"})
    if len(avail) == len(MODELS):
        e = merged[MODELS[0]][["image_path", "label", "component_id", "outer_fold"]].copy()
        for c in COND: e[f"p_{c}"] = np.mean([merged[k].set_index("image_path").loc[e.image_path, f"p_{c}"].values for k in MODELS], axis=0)
        merged["ensemble"] = e
    rows, folds = [], []
    for k, d in merged.items():
        d.to_csv(ROUND3 / "predictions" / f"masking_oof_{k}.csv", index=False)
        for cname in ["detected sector removed", "detected sector only"]:
            r = paired_boot(d.label, d["p_full image"], d[f"p_{cname}"], d.component_id)
            rows.append(dict(model=DISPLAY[k], condition=cname, n_images=len(d), n_cases=d.component_id.nunique(),
                             auroc_full=r["auc_a"], auroc_full_lo=r["ci_a"][0], auroc_full_hi=r["ci_a"][1],
                             auroc_condition=r["auc_b"], auroc_condition_lo=r["ci_b"][0], auroc_condition_hi=r["ci_b"][1],
                             delta=r["delta"], delta_lo=r["ci_delta"][0], delta_hi=r["ci_delta"][1], bootstrap_p=r["p"], n_boot=r["n_boot"]))
        for cname in COND:
            for f in sorted(d.outer_fold.unique()):
                x = d[d.outer_fold == f]; folds.append(dict(model=DISPLAY[k], condition=cname, fold=int(f), auroc=roc_auc_score(x.label, x[f"p_{cname}"])))
    MA = pd.DataFrame(rows); MA.to_csv(ROUND3 / "tables" / "MA1_masking_paired.csv", index=False)
    pd.DataFrame(folds).to_csv(ROUND3 / "tables" / "MA2_masking_folds.csv", index=False)
    show = MA.assign(full=MA.apply(lambda r: fmt3(r.auroc_full, r.auroc_full_lo, r.auroc_full_hi), axis=1),
                     restricted=MA.apply(lambda r: fmt3(r.auroc_condition, r.auroc_condition_lo, r.auroc_condition_hi), axis=1),
                     delta_auroc=MA.apply(lambda r: fmt3(r.delta, r.delta_lo, r.delta_hi), axis=1))[["model", "condition", "n_cases", "full", "restricted", "delta_auroc", "bootstrap_p"]]
    save_table(show.round(4), "MA1_masking_summary", "Masking control: pooled out-of-fold AUROC and paired ΔAUROC versus full image (case-clustered 95% CI)",
               note="No equivalence margin was specified; ΔAUROC intervals are reported as estimated, not as evidence of equivalence.")

## Stage SD — Split design: paired ΔAUROC, image-level minus case-grouped (editor points 13, 28, 29)

In [ ]:
with stage("SD", "Split-design paired analysis") as go:
  if go:
    rows, fm_rows = [], []
    for k in ["convnext_small", "efficientnet_b3"]:
        gp = CV_DIR / "cytology" / k / "oof_predictions.csv"; ip = ROUND2 / "imagelevel_cv" / "cytology" / k / "oof_predictions.csv"
        if not (gp.exists() and ip.exists()): print("missing", k); continue
        g = pd.read_csv(gp, low_memory=False); i = pd.read_csv(ip, low_memory=False)
        lg, li = fold_leak_share(g), fold_leak_share(i)
        m = g[["image_path", "label", "component_id", "prob_malignant"]].merge(i[["image_path", "label", "prob_malignant"]].rename(columns={"label": "label_i", "prob_malignant": "p_image"}),
                                                                             on="image_path", validate="one_to_one")
        assert len(m) == len(g) == len(i) and (m.label == m.label_i).all()
        r = paired_boot(m.label, m.prob_malignant, m.p_image, m.component_id)
        fg = pd.read_csv(CV_DIR / "cytology" / k / "fold_metrics.csv"); fi = pd.read_csv(ROUND2 / "imagelevel_cv" / "cytology" / k / "fold_metrics.csv")
        rows.append(dict(model=DISPLAY[k], n_images=len(m), n_cases=m.component_id.nunique(), within_cv_case_leakage_grouped=lg, within_cv_case_leakage_image=li,
                         auroc_grouped=r["auc_a"], auroc_grouped_lo=r["ci_a"][0], auroc_grouped_hi=r["ci_a"][1],
                         auroc_image=r["auc_b"], auroc_image_lo=r["ci_b"][0], auroc_image_hi=r["ci_b"][1],
                         delta=r["delta"], delta_lo=r["ci_delta"][0], delta_hi=r["ci_delta"][1], bootstrap_p=r["p"],
                         fold_mean_grouped=fg.auroc.mean(), fold_mean_image=fi.auroc.mean()))
        m.to_csv(ROUND3 / "predictions" / f"splitdesign_oof_{k}.csv", index=False)
    SD = pd.DataFrame(rows); SD.to_csv(ROUND3 / "tables" / "SD1_split_design_paired.csv", index=False)
    save_table(SD.round(4), "SD1_split_design_paired", "Split design (cytology): pooled out-of-fold AUROC under case-grouped and image-level folds, paired ΔAUROC (image-level − grouped)",
               note="Each image has an out-of-fold prediction under both designs, so the pooled comparison is paired at image level and resampled by case. Fold k contains different images under the two designs, so fold-wise differences are not paired.")

## Stage MD — Metadata-only shortcut baseline and class-wise acquisition metadata (editor points 7, 8, 41)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
MD_MODELS = {"logistic regression": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced")),
             "gradient boosting": lambda: make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED))}
PRIMARY_SET, PRIMARY_MODEL = "B: A + border strips (overlays)", "logistic regression"

def load_meta():
    cache = ROUND3 / "predictions" / "metadata_features.csv"
    if cache.exists(): return pd.read_csv(cache, low_memory=False)
    fm = read_frozen(); rows = []
    for r in fm.itertuples():
        f = meta_features(r.image_path); f.update(image_path=r.image_path, case_key=r.case_key, modality=r.modality, label=int(r.label), split=r.split, component_id=r.component_id)
        rows.append(f)
    F = pd.DataFrame(rows); F.to_csv(cache, index=False); return F

with stage("MD", "Metadata-only baseline") as go:
  if go:
    F = load_meta(); print("metadata features:", F.shape)
    med = lambda s: f"{s.median():.3g} [{s.quantile(.25):.3g}–{s.quantile(.75):.3g}]"
    cls = []
    for (mo, lab), g in F.groupby(["modality", "label"]):
        top = g.assign(fr=g.width.astype(int).astype(str) + "×" + g.height.astype(int).astype(str)).fr.value_counts()
        cls.append({"modality": mo, "class": "PTC" if lab else "benign", "images": len(g), "width px": med(g.width), "height px": med(g.height), "aspect ratio": med(g.aspect),
                    "colour content (% images)": f"{100 * (g.colored_frac > 0.001).mean():.0f}", "near-black fraction": med(g.near_black),
                    "coloured-pixel fraction": med(g.colored_frac), "top-strip near-white fraction": med(g.top_near_white),
                    "upper-case file extension (%)": f"{100 * g.ext_uppercase.mean():.0f}", "EXIF present (%)": f"{100 * g.exif_present.mean():.0f}",
                    "most common frame (n)": f"{top.index[0]} ({top.iloc[0]})", "distinct frame sizes": int(top.size)})
    save_table(pd.DataFrame(cls), "MD1_classwise_metadata", "Acquisition metadata by modality and class: median [IQR] or %")
    us = F[F.modality == "ultrasound"].assign(frame=lambda d: d.width.astype(int).astype(str) + "×" + d.height.astype(int).astype(str))
    fr = us.groupby(["frame", "label"]).size().unstack(fill_value=0).rename(columns={0: "benign", 1: "PTC"}); fr["total"] = fr.sum(1)
    fr = fr.sort_values("total", ascending=False).reset_index(); fr["PTC share"] = (fr.PTC / fr.total).round(2)
    save_table(fr.head(10), "MD2_ultrasound_frame_sizes", "Ultrasound frame sizes by class (10 most frequent)")

    rows, coefs, roc_store = [], [], {}
    for mo in ["ultrasound", "cytology"]:
        d = F[F.modality == mo].copy()
        fa = pd.read_csv(ROUND2 / "manifests" / f"cv_fold_assignment_{mo}.csv"); d["outer_fold"] = d.image_path.map(dict(zip(fa.image_path, fa.outer_fold)))
        tr, va, te = d[d.split == "train"], d[d.split == "val"], d[d.split == "test"]; dev = d[d.split != "test"]
        assert dev.outer_fold.notna().all(), "fold assignment does not cover the development set"
        for sname, cols in FEATURE_SETS.items():
            for mname, factory in MD_MODELS.items():
                clf = factory().fit(tr[cols], tr.label); thr = youden(va.label, clf.predict_proba(va[cols])[:, 1]); pt = clf.predict_proba(te[cols])[:, 1]
                m = full_metrics(te.label, pt, thr); ci = boot_ci(te.label, pt, te.component_id, thr)
                oof = np.zeros(len(dev))
                for f in sorted(dev.outer_fold.unique()):
                    trn, hold = dev.outer_fold != f, dev.outer_fold == f
                    oof[hold.values] = factory().fit(dev[trn][cols], dev[trn].label).predict_proba(dev[hold][cols])[:, 1]
                cvm = roc_auc_score(dev.label, oof); cvci = boot_ci(dev.label, oof, dev.component_id)
                rows.append(dict(modality=mo, feature_set=sname, model=mname, test_n=len(te), test_auroc=m["auroc"], test_auroc_lo=ci["auroc"][0], test_auroc_hi=ci["auroc"][1],
                                 test_auprc=m["auprc"], no_skill=m["prevalence"], threshold_val=thr, test_sensitivity=m["sensitivity"], test_specificity=m["specificity"],
                                 cv_n=len(dev), cv_pooled_auroc=cvm, cv_auroc_lo=cvci["auroc"][0], cv_auroc_hi=cvci["auroc"][1]))
                if sname == PRIMARY_SET and mname == PRIMARY_MODEL:
                    roc_store[mo] = dict(y=te.label.values, p=pt)
                    pd.DataFrame({"image_path": te.image_path, "label": te.label, "prob_metadata_only": pt}).to_csv(ROUND3 / "predictions" / f"metadata_only_test_{mo}.csv", index=False)
                    lr = clf.named_steps["logisticregression"]
                    for c, w in zip(cols, lr.coef_[0]): coefs.append(dict(modality=mo, feature=c, standardised_coefficient=w))
    MD3 = pd.DataFrame(rows); MD3.to_csv(ROUND3 / "tables" / "MD3_metadata_baseline.csv", index=False)
    show = MD3.assign(test=MD3.apply(lambda r: fmt3(r.test_auroc, r.test_auroc_lo, r.test_auroc_hi), axis=1),
                      grouped_cv=MD3.apply(lambda r: fmt3(r.cv_pooled_auroc, r.cv_auroc_lo, r.cv_auroc_hi), axis=1))[["modality", "feature_set", "model", "test", "grouped_cv", "test_sensitivity", "test_specificity"]]
    save_table(show.round(3), "MD3_metadata_baseline_summary", "Metadata-only shortcut diagnostic (no CNN): frozen test AUROC and grouped-CV pooled AUROC (case-clustered 95% CI)",
               note="A shortcut diagnostic, not a clinical model. Primary specification: logistic regression on feature set B, fitted on the frozen training partition with the threshold chosen on validation.")
    C = pd.DataFrame(coefs); C["abs"] = C.standardised_coefficient.abs()
    save_table(C.sort_values(["modality", "abs"], ascending=[True, False]).groupby("modality").head(8).drop(columns="abs").round(3), "MD4_metadata_top_coefficients",
               "Largest standardised coefficients of the primary metadata-only model")
    (ROUND3 / "tables" / "MD5_roc_source.json").write_text(json.dumps({k: {"y": v["y"].tolist(), "p": v["p"].tolist()} for k, v in roc_store.items()}))

## Stage ND — Near-duplicate adjudication (editor point 9)

In [ ]:
with stage("ND", "Near-duplicate adjudication") as go:
  if go:
    may = pd.read_csv(MAY_SPLIT, low_memory=False); fm = read_frozen(); fsplit = dict(zip(fm.image_path, fm.split))
    rows = []
    for mo in ["cytology", "ultrasound"]:
        d = may[may.modality == mo].reset_index(drop=True)
        pt = pd.read_csv(ROUND2 / "tables" / f"A4b_phash_pairs_{mo}.csv")
        cand = pt[pt.cross_frozen_partition & (pt.hamming <= 6)]
        for _, r in cand.iterrows():
            a, b = d.loc[int(r.i)], d.loc[int(r.j)]; ga, gb = _gray256(a.image_path), _gray256(b.image_path)
            s = ssim256(ga, gb); corr = float(np.corrcoef(ga.ravel(), gb.ravel())[0, 1])
            rows.append(dict(modality=mo, hamming=int(r.hamming), case_a=case_key_from_path(a.image_path), file_a=Path(a.image_path).name, split_a=fsplit.get(a.image_path, "excluded"),
                             case_b=case_key_from_path(b.image_path), file_b=Path(b.image_path).name, split_b=fsplit.get(b.image_path, "excluded"),
                             same_label=bool(a.label == b.label), md5_identical=bool(a.md5 == b.md5), ssim_256=round(s, 4), pearson_r=round(corr, 4),
                             verdict="near-duplicate (SSIM ≥ 0.90)" if s >= 0.90 else "not a near-duplicate (SSIM < 0.90)", path_a=a.image_path, path_b=b.image_path))
    ND = pd.DataFrame(rows)
    save_table(ND.drop(columns=["path_a", "path_b"]), "ND1_near_duplicate_adjudication",
               "Every perceptual-hash pair (Hamming ≤ 6) whose members fall in different frozen partitions, adjudicated by structural similarity",
               note="Criterion fixed in code before the similarity values were computed: a pair is a near-duplicate if SSIM ≥ 0.90 on 256 × 256 greyscale versions.")
    if len(ND):
        fig, axes = plt.subplots(len(ND), 2, figsize=(5.0, 2.2 * len(ND))); axes = np.atleast_2d(axes)
        for i, r in ND.reset_index(drop=True).iterrows():
            for j, (pth, ckey, sp) in enumerate([(r.path_a, r.case_a, r.split_a), (r.path_b, r.case_b, r.split_b)]):
                ax = axes[i, j]; ax.imshow(Image.open(pth).convert("L"), cmap="gray"); ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(f"{r.modality} {ckey} [{sp}]" + (f"\nHamming {r.hamming} · SSIM {r.ssim_256:.2f} · {r.verdict.split(' (')[0]}" if j == 0 else ""), fontsize=6.5)
        fig.tight_layout(); save_fig(fig, "S_near_duplicate_adjudication", vector=False)
    test_side = set()
    for _, r in ND.iterrows():
        if r.split_a == "test" and r.split_b != "test": test_side.add(r.case_a)
        if r.split_b == "test" and r.split_a != "test": test_side.add(r.case_b)
    sens = []
    for mo in ["cytology", "ultrasound"]:
        t = pd.read_csv(EXP1B_DIR / mo / "ensemble" / "test_predictions.csv"); t["case_key"] = t.image_path.map(case_key_from_path)
        nd_cases = set(ND[(ND.modality == mo)].pipe(lambda x: pd.concat([x.case_a, x.case_b])).tolist()) & test_side
        for scen, drop in [("as analysed", set()), ("excluding test-side cases of all candidate pairs", nd_cases),
                           ("excluding test-side cases adjudicated near-duplicate", nd_cases & set(ND[ND.verdict.str.startswith("near")].pipe(lambda x: pd.concat([x.case_a, x.case_b])).tolist()))]:
            x = t[~t.case_key.isin(drop)]; g = x.groupby("case_key").agg(label=("label", "first"), p=("prob_malignant", "mean")).reset_index()
            ci_i = boot_ci(x.label, x.prob_malignant, x.case_key); ci_c = boot_ci(g.label, g.p, g.case_key)
            sens.append(dict(modality=mo, scenario=scen, excluded_cases=", ".join(sorted(drop)) or "none", test_images=len(x), test_cases=len(g),
                             image_level_auroc=fmt3(roc_auc_score(x.label, x.prob_malignant), *ci_i["auroc"]), case_level_auroc=fmt3(roc_auc_score(g.label, g.p), *ci_c["auroc"])))
    save_table(pd.DataFrame(sens), "ND2_near_duplicate_sensitivity", "Frozen-test ensemble results with test-side members of partition-crossing pairs excluded")

## Stage HX — Systematically selected acquisition examples (editor point 42)

In [ ]:
with stage("HX", "Acquisition examples") as go:
  if go:
    F = load_meta(); us = F[F.modality == "ultrasound"].assign(frame=lambda d: d.width.astype(int).astype(str) + "×" + d.height.astype(int).astype(str))
    frames = us.frame.value_counts().head(4).index.tolist()
    fig, axes = plt.subplots(2, len(frames), figsize=(2.3 * len(frames), 4.4))
    for j, fr_ in enumerate(frames):
        g = us[us.frame == fr_]; nb, npc = int((g.label == 0).sum()), int((g.label == 1).sum())
        for i, (lab, name) in enumerate([(0, "benign"), (1, "PTC")]):
            ax = axes[i, j]; ax.set_xticks([]); ax.set_yticks([]); s = g[g.label == lab]
            if len(s):
                r = s.sample(1, random_state=SEED).iloc[0]; ax.imshow(Image.open(r.image_path).convert("RGB"))
                ax.set_title(f"{name} · {r.case_key}", fontsize=7)
            else:
                ax.text(0.5, 0.5, f"no {name} image\nwith this frame", ha="center", va="center", fontsize=7); ax.set_frame_on(False)
            if i == 0: ax.text(0.5, 1.22, f"frame {fr_}\n{nb} benign / {npc} PTC", transform=ax.transAxes, ha="center", fontsize=7.5)
    fig.suptitle("Acquisition context: the four most frequent ultrasound frame sizes; one randomly drawn image per class (fixed seed)", fontsize=8, y=1.02)
    fig.tight_layout(); save_fig(fig, "S_acquisition_examples", vector=False)

## Stage GC — Grad-CAM on one shared scale (editor points 33–35)

In [ ]:
GPU_TASKS_R3 = ROUND3 / "scripts" / "r3_gpu_tasks.py"
_r3src = GPU_TASKS_SRC.replace("cams = np.zeros((len(df), S, S), np.float16)", "cams = np.zeros((len(df), S, S), np.float32)")
_r3src = _r3src.replace("cams[i] = (cam / cam.max() if cam.max() > 0 else cam).astype(np.float16)", "cams[i] = cam.astype(np.float32)  # ROUND3: raw maps, one shared scale")
_r3src = _r3src.replace('np.savez_compressed(outdir / "cams.npz", cams=cams)', 'np.savez_compressed(outdir / "cams_raw.npz", cams=cams)')
assert _r3src.count("ROUND3: raw maps") == 1 and "cams_raw.npz" in _r3src
GPU_TASKS_R3.write_text(_r3src, encoding="utf-8")

def gradcam_panels(g, cams, idx_of, vmax, sel, name, ncols):
    nrows = int(np.ceil(len(sel) / ncols)); fig, axes = plt.subplots(nrows, ncols, figsize=(2.1 * ncols + 0.6, 2.35 * nrows)); axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for ax, (_, r) in zip(axes, sel.iterrows()):
        ax.axis("on"); base = np.asarray(Image.open(r.image_path).convert("L").resize((224, 224)), float)
        ax.imshow(base, cmap="gray"); hm = ax.imshow(cams[idx_of[r.image_path]], cmap="magma", alpha=0.45, vmin=0, vmax=vmax)
        fb = r.box_method != "detected"
        ax.add_patch(Rectangle((r.box_x0 * 224, r.box_y0 * 224), (r.box_x1 - r.box_x0) * 224, (r.box_y1 - r.box_y0) * 224, fill=False, lw=1, ls="--" if fb else "-", ec="#f0a030" if fb else "#40c0ff"))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{r.outcome} · true {'PTC' if r.label else 'benign'} · p = {r.prob_malignant:.2f}\n{'fallback box' if fb else 'heuristically detected sonographic region'}", fontsize=6.3)
    fig.colorbar(hm, ax=list(axes), fraction=0.02, pad=0.01, label="Grad-CAM (raw, shared scale)")
    save_fig(fig, name, vector=False)

with stage("GC", "Grad-CAM, shared scale") as go:
  if go:
    gm_path = ROUND2 / "manifests" / "gradcam_manifest.csv"
    if not (ROUND3 / "gradcam" / "cams_raw.npz").exists():
        run_cmd([TORCH_PY, GPU_TASKS_R3, "gradcam", "--script03b", SCRIPT_03B, "--manifest", gm_path, "--exp1b-dir", EXP1B_DIR,
                 "--model", "efficientnet_b3", "--out", ROUND3 / "gradcam"], "GC_gradcam.log")
    gm = pd.read_csv(gm_path); cams = np.load(ROUND3 / "gradcam" / "cams_raw.npz")["cams"]; idx_of = {p: i for i, p in enumerate(gm.image_path)}
    thr = float(json.loads((EXP1B_DIR / "ultrasound" / "efficientnet_b3" / "metrics.json").read_text())["image_level_threshold_from_val"])
    g = gm.copy(); g["pred"] = (g.prob_malignant >= thr).astype(int)
    g["outcome"] = np.select([(g.label == 1) & (g.pred == 1), (g.label == 0) & (g.pred == 0), (g.label == 0) & (g.pred == 1)], ["TP", "TN", "FP"], "FN")
    pos = cams[cams > 0]; vmax = float(np.percentile(pos, 99)) if pos.size else 1.0
    t = g[g.partition == "test"]
    rep = pd.concat([x.iloc[[(x.prob_malignant - x.prob_malignant.median()).abs().argsort().iloc[0]]] for o in ["TP", "TN", "FP", "FN"] for x in [t[t.outcome == o]] if len(x)])
    gradcam_panels(g, cams, idx_of, vmax, rep, "S_gradcam_representative_4", 4)
    two = pd.concat([x.sort_values("prob_malignant").iloc[[0, -1]] if len(x) > 1 else x for o in ["TP", "TN", "FP", "FN"] for x in [t[t.outcome == o]]])
    gradcam_panels(g, cams, idx_of, vmax, two, "S_gradcam_8", 4)
    note_md(f"- Grad-CAM panels: one shared colour scale (0 to the 99th percentile of positive raw attribution, {vmax:.3g}); magma colour map; representative case = median probability within each outcome.")

## Stage FIG — Figures (editor points 21–42, 51–55)

In [ ]:
def counts():
    fm = read_frozen(); may = pd.read_csv(MAY_SPLIT, low_memory=False); may["case_key"] = may.image_path.map(case_key_from_path)
    c = {}
    for nm, d in [("rel", may), ("ana", fm)]:
        for mo in ["ultrasound", "cytology"]:
            x = d[d.modality == mo]; c[f"{nm}_{mo}_img"] = len(x); c[f"{nm}_{mo}_cases"] = x.case_key.nunique()
            c[f"{nm}_{mo}_ben"] = x[x.label == 0].case_key.nunique(); c[f"{nm}_{mo}_ptc"] = x[x.label == 1].case_key.nunique()
    for mo in ["ultrasound", "cytology"]:
        for sp in ["train", "val", "test"]:
            x = fm[(fm.modality == mo) & (fm.split == sp)]; c[f"{mo}_{sp}_img"] = len(x); c[f"{mo}_{sp}_cases"] = x.case_key.nunique()
        x = fm[(fm.modality == mo) & (fm.split != "test")]; c[f"{mo}_dev_img"] = len(x); c[f"{mo}_dev_cases"] = x.case_key.nunique()
    dup = pd.read_csv(ROUND2 / "tables" / "A2b_duplicate_summary.csv")
    c["dup_groups"] = int(dup.groups.sum()); c["dup_files"] = int(dup.images.sum())
    cl = dup[dup.conflict_type == "cross-label"]; c["conf_groups"] = int(cl.groups.sum()); c["conf_files"] = int(cl.images.sum())
    c["excluded"] = len(pd.read_csv(EXCLUDED_CSV))
    for nm in ["TN3K_test", "TN3K_trainval", "DDTI_original"]:
        f = ROUND2 / "manifests" / f"external_{nm}.csv"
        if f.exists():
            e = pd.read_csv(f); c[f"{nm}_img"] = len(e); c[f"{nm}_cases"] = e.case_id.nunique()
            if nm.startswith("TN3K"): c[f"{nm}_ben"], c[f"{nm}_mal"] = int((e.label == 0).sum()), int((e.label == 1).sum())
            else:
                v = e[e.y_primary.isin([0, 1])]; c["DDTI_tirads_img"], c["DDTI_tirads_cases"] = len(v), v.case_id.nunique()
    return c

def fbox(ax, x, y, w, h, title, body, fc, ec, ls="-", fs=6.3):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.004,rounding_size=0.01", fc=fc, ec=ec, lw=1.0, ls=ls))
    ax.text(x + w / 2, y + h - 0.012, title, ha="center", va="top", fontsize=fs + 0.6, fontweight="bold")
    ax.text(x + w / 2, y + h - 0.045, body, ha="center", va="top", fontsize=fs, linespacing=1.3)
def farrow(ax, x0, y0, x1, y1, ls="-", col="#333333"):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", lw=0.9, color=col, ls=ls))

def fig1(c):
    fig, ax = plt.subplots(figsize=(7.2, 5.2)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    for y, h, t, fc in [(0.69, 0.30, "1  Archive audit", "#f2f2f2"), (0.365, 0.30, "2  Internal evaluation", "#eaf1f8"), (0.02, 0.32, "3  Post-exposure external reassessment", "#f1f0f6")]:
        ax.add_patch(Rectangle((0.0, y), 1.0, h, fc=fc, ec="none")); ax.text(0.012, y + h - 0.014, t, fontsize=8, fontweight="bold", va="top")
    W, Y, H = 0.182, 0.715, 0.205; xs = [0.012 + i * (W + 0.0165) for i in range(5)]
    A = [("Released archive", f"Ultrasound\n{c['rel_ultrasound_img']} images / {c['rel_ultrasound_cases']} cases\nCytology\n{c['rel_cytology_img']:,} blocks / {c['rel_cytology_cases']} cases"),
         ("Exact duplicates", f"{c['dup_groups']} MD5 groups\n{c['dup_files']} files"),
         ("Contradictory labels", f"{c['conf_groups']} ultrasound groups\n({c['conf_files']} files) filed\nas both benign and PTC"),
         ("Excluded", f"{c['excluded']} ultrasound images\n(identical across labels\nby MD5 or pHash)"),
         ("Analysed archive", f"Ultrasound\n{c['ana_ultrasound_img']} images / {c['ana_ultrasound_cases']} cases\nCytology\n{c['ana_cytology_img']:,} blocks / {c['ana_cytology_cases']} cases")]
    for i, (t, b_) in enumerate(A):
        fbox(ax, xs[i], Y, W, H, t, b_, "#ffffff", "#7a7a7a", fs=5.7)
        if i < 4: farrow(ax, xs[i] + W + 0.001, Y + H / 2, xs[i + 1] - 0.001, Y + H / 2)
    W2, Y2, H2 = 0.31, 0.39, 0.215; xs2 = [0.012, 0.345, 0.678]
    B = [("Frozen case-level partition", f"Cytology: train {c['cytology_train_cases']} cases / {c['cytology_train_img']} blocks\nval {c['cytology_val_cases']} / {c['cytology_val_img']} · test {c['cytology_test_cases']} / {c['cytology_test_img']}\nUltrasound: train {c['ultrasound_train_cases']} · val {c['ultrasound_val_cases']} · test {c['ultrasound_test_cases']}\n(one image per case)", "#ffffff"),
         ("Grouped 5-fold CV (development)", f"Cytology: {c['cytology_dev_cases']} cases / {c['cytology_dev_img']:,} blocks\nUltrasound: {c['ultrasound_dev_cases']} cases / images\nthresholds from inner validation", "#ffffff"),
         ("Frozen test (held out)", f"Cytology: {c['cytology_test_cases']} cases / {c['cytology_test_img']} blocks\nUltrasound: {c['ultrasound_test_cases']} cases / images\nvalidation-selected thresholds", "#dbe7f3")]
    for i, (t, b_, fc) in enumerate(B):
        fbox(ax, xs2[i], Y2, W2, H2, t, b_, fc, COL["cyto"], fs=6.1)
    farrow(ax, xs2[0] + W2 + 0.001, Y2 + H2 / 2, xs2[1] - 0.001, Y2 + H2 / 2); farrow(ax, xs2[1] + W2 + 0.001, Y2 + H2 / 2, xs2[2] - 0.001, Y2 + H2 / 2)
    farrow(ax, xs[4] + W / 2, Y, xs2[2] + W2 / 2, Y2 + H2 + 0.002)
    Y3, H3 = 0.05, 0.215
    C3 = [("TN3K (distributed labels)", f"test {c.get('TN3K_test_img', 0)} images\n({c.get('TN3K_test_ben', 0)} benign / {c.get('TN3K_test_mal', 0)} malignant)\ntrainval {c.get('TN3K_trainval_img', 0):,} images\nno patient IDs: image-level CIs"),
          ("DDTI (original release)", f"{c.get('DDTI_original_img', 0)} images / {c.get('DDTI_original_cases', 0)} cases\nwith TI-RADS 2–5: {c.get('DDTI_tirads_img', 0)} / {c.get('DDTI_tirads_cases', 0)}\nendpoint: TI-RADS suspicion\ncategory, not malignancy"),
          ("Status of these analyses", "Both datasets were accessed\nduring earlier analytical\ndevelopment; results are a\npost-exposure reassessment")]
    for i, (t, b_) in enumerate(C3):
        fbox(ax, xs2[i], Y3, W2, H3, t, b_, "#ffffff", COL["ext"], ls="--", fs=6.1)
    farrow(ax, xs2[2] + W2 / 2, Y2, xs2[1] + W2 / 2, Y3 + H3 + 0.002, ls="--", col=COL["ext"])
    ax.text(0.99, 0.355, "frozen ultrasound models applied without retraining", ha="right", va="top", fontsize=5.8, color=COL["ext"], style="italic")
    save_fig(fig, "Fig1_provenance_evidence_hierarchy")

def forest(ax, items, xlim=(0.4, 1.0), xlabel="Pooled out-of-fold AUROC (95% CI)"):
    for i, it in enumerate(items[::-1]):
        y = i; col = MODEL_COL.get(it["key"], "#555555")
        if it.get("folds") is not None:
            ax.scatter(it["folds"], np.full(len(it["folds"]), y) + np.linspace(-0.12, 0.12, len(it["folds"])), s=6, color="#bdbdbd", zorder=1, lw=0)
        ax.plot([it["lo"], it["hi"]], [y, y], color=col, lw=1.6, zorder=2)
        ax.scatter([it["est"]], [y], s=34 if it["key"] == "ensemble" else 22, marker="D" if it["key"] == "ensemble" else "o", color=col, zorder=3)
        ax.text(1.02, y, f"{it['est']:.3f} ({it['lo']:.3f}–{it['hi']:.3f})", va="center", fontsize=6, transform=ax.get_yaxis_transform())
    ax.set_yticks(range(len(items))); ax.set_yticklabels([it["label"] for it in items[::-1]]); ax.set_xlim(*xlim)
    ax.axvline(0.5, color=COL["chance"], lw=0.8, ls="--"); ax.set_xlabel(xlabel)

def roc_panel(ax, curves, title, legend_title=None):
    for c_ in curves:
        fpr, tpr, _ = roc_curve(c_["y"], c_["p"]); ax.plot(fpr, tpr, color=MODEL_COL.get(c_["key"], "#555"), lw=1.8 if c_["key"] == "ensemble" else 1.1, label=c_["label"])
    ax.plot([0, 1], [0, 1], ls="--", color=COL["chance"], lw=0.8); ax.set_xlabel("1 − specificity"); ax.set_ylabel("Sensitivity"); ax.set_title(title, loc="left", fontsize=8)
    ax.set_aspect("equal")
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=6.2, title=legend_title, title_fontsize=5.8)

def fig2():
    C1 = pd.read_csv(ROUND2 / "tables" / "C1_cv_summary.csv"); C2 = pd.read_csv(ROUND2 / "tables" / "C2_cv_fold_level.csv"); F2 = pd.read_csv(ROUND2 / "tables" / "F2_frozen_test_case_level.csv")
    inv = {v: k for k, v in DISPLAY.items()}
    fig = plt.figure(figsize=(7.2, 7.0)); gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2], hspace=0.62, wspace=1.05)
    for j, (mo, name) in enumerate([("cytology", "A  Cytology — grouped 5-fold CV"), ("ultrasound", "B  Ultrasound — grouped 5-fold CV")]):
        ax = fig.add_subplot(gs[0, j]); items = []
        for _, r in C1[C1.modality == mo].iterrows():
            est, lo, hi = parse_ci(r.pooled_oof_auroc); k = inv.get(r.model, r.model)
            items.append(dict(label=r.model.replace(" (mean of 5)", ""), key=k, est=est, lo=lo, hi=hi, folds=C2[(C2.modality == mo) & (C2.model == r.model)].auroc.values))
        forest(ax, items, xlim=(0.4, 1.0)); ax.set_title(name, loc="left", fontsize=8)
    for j, (mo, name) in enumerate([("cytology", "C  Cytology — frozen test, case level"), ("ultrasound", "D  Ultrasound — frozen test (1 image per case)")]):
        ax = fig.add_subplot(gs[1, j]); curves = []
        f2 = F2[F2.modality == mo]; cv = C1[(C1.modality == mo) & (C1.model != DISPLAY["ensemble"])].copy()
        cv["est"] = cv.pooled_oof_auroc.map(lambda s: parse_ci(s)[0]); best = inv[cv.sort_values("est").iloc[-1].model]
        for k in ["ensemble", best]:
            stem = "test_source_group_predictions.csv"
            t = pd.read_csv(EXP1B_DIR / mo / k / stem); a = f2[f2.model == DISPLAY[k]].auroc.iloc[0]
            curves.append(dict(y=t.label.values, p=t.prob_malignant.values, key=k, label=f"{DISPLAY[k].replace(' (mean of 5)', '')}: {a.replace('–', '–')}"))
        roc_panel(ax, curves, name)
    save_fig(fig, "Fig2_internal_case_level_performance")

def fig3():
    SD = pd.read_csv(ROUND3 / "tables" / "SD1_split_design_paired.csv")
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 1.9), gridspec_kw=dict(width_ratios=[1, 1.1], wspace=0.9))
    ax = axes[0]
    for i, r in SD.reset_index(drop=True).iterrows():
        y = len(SD) - 1 - i
        for est, lo, hi, dy, mk, lab in [(r.auroc_grouped, r.auroc_grouped_lo, r.auroc_grouped_hi, 0.12, "o", "case-grouped folds"), (r.auroc_image, r.auroc_image_lo, r.auroc_image_hi, -0.12, "s", "image-level folds")]:
            ax.plot([lo, hi], [y + dy] * 2, color=MODEL_COL[{v: k for k, v in DISPLAY.items()}[r.model]], lw=1.3)
            ax.scatter([est], [y + dy], marker=mk, s=20, color=MODEL_COL[{v: k for k, v in DISPLAY.items()}[r.model]], label=lab if i == 0 else None, zorder=3)
    ax.set_yticks(range(len(SD))); ax.set_yticklabels(SD.model[::-1]); ax.set_xlabel("Pooled out-of-fold AUROC (95% CI)"); ax.set_title("A  Cytology, tested backbones", loc="left", fontsize=8)
    ax.legend(frameon=False, fontsize=6, loc="upper center", bbox_to_anchor=(0.5, -0.36), ncol=2); ax.set_xlim(0.8, 1.0); ax.set_ylim(-0.6, len(SD) - 0.4)
    ax = axes[1]
    for i, r in SD.reset_index(drop=True).iterrows():
        y = len(SD) - 1 - i; col = MODEL_COL[{v: k for k, v in DISPLAY.items()}[r.model]]
        ax.plot([r.delta_lo, r.delta_hi], [y, y], color=col, lw=1.6); ax.scatter([r.delta], [y], color=col, s=22, zorder=3)
        ax.text(1.03, y, f"{r.delta:+.3f} ({r.delta_lo:+.3f} to {r.delta_hi:+.3f})\nwithin-CV case leakage {100 * r.within_cv_case_leakage_grouped:.0f}% → {100 * r.within_cv_case_leakage_image:.1f}%", va="center", fontsize=6, transform=ax.get_yaxis_transform())
    ax.axvline(0, color=COL["chance"], lw=0.8, ls="--"); ax.set_xlim(-0.06, 0.06); ax.set_yticks(range(len(SD))); ax.set_yticklabels(SD.model[::-1]); ax.set_ylim(-0.6, len(SD) - 0.4)
    ax.set_xlabel("ΔAUROC, image-level − case-grouped (paired, case-clustered 95% CI)"); ax.set_title("B  Paired difference", loc="left", fontsize=8)
    save_fig(fig, "Fig3_split_design_paired")

def fig4():
    MA = pd.read_csv(ROUND3 / "tables" / "MA1_masking_paired.csv"); ex = json.loads((ROUND3 / "tables" / "Q4_main_figure_examples.json").read_text())
    Q = pd.read_csv(ROUND3 / "tables" / "Q0_masking_qa_per_image.csv").set_index("image_path")
    models = list(dict.fromkeys(MA.model)); inv = {v: k for k, v in DISPLAY.items()}; short = lambda m: m.replace(" (mean of 5)", "")
    mk = {"full image": "o", "detected sector removed": "s", "detected sector only": "^"}
    fig = plt.figure(figsize=(7.2, 6.6)); gs = fig.add_gridspec(2, 3, height_ratios=[1.35, 1], hspace=0.62, wspace=0.75)
    ax = fig.add_subplot(gs[0, 0]); offs = {"full image": 0.24, "detected sector removed": 0.0, "detected sector only": -0.24}
    for i, m in enumerate(models):
        y = len(models) - 1 - i; sub = MA[MA.model == m]; col = MODEL_COL[inv[m]]; r0 = sub.iloc[0]
        vals = {"full image": (r0.auroc_full, r0.auroc_full_lo, r0.auroc_full_hi)}
        for _, r in sub.iterrows(): vals[r.condition] = (r.auroc_condition, r.auroc_condition_lo, r.auroc_condition_hi)
        for cnd, (e, lo, hi) in vals.items():
            ax.plot([lo, hi], [y + offs[cnd]] * 2, color=col, lw=1.0); ax.scatter([e], [y + offs[cnd]], marker=mk[cnd], color=col, s=14, zorder=3)
    ax.axvline(0.5, color=COL["chance"], lw=0.8, ls="--"); ax.set_yticks(range(len(models))); ax.set_yticklabels([short(m) for m in models[::-1]]); ax.set_xlim(0.35, 0.95)
    ax.set_xlabel("Pooled OOF AUROC (95% CI)"); ax.set_title("A  Ultrasound: absolute performance", loc="left", fontsize=8)
    handles = [plt.Line2D([], [], marker=mk[c_], ls="", color="#444", label=c_) for c_ in mk]
    ax.legend(handles=handles, frameon=False, fontsize=6, loc="upper center", bbox_to_anchor=(0.5, -0.2))
    ax = fig.add_subplot(gs[0, 1:3])
    for i, m in enumerate(models):
        y = len(models) - 1 - i; col = MODEL_COL[inv[m]]
        for cnd, dy in [("detected sector removed", 0.16), ("detected sector only", -0.16)]:
            r = MA[(MA.model == m) & (MA.condition == cnd)]
            if not len(r): continue
            r = r.iloc[0]; ax.plot([r.delta_lo, r.delta_hi], [y + dy] * 2, color=col, lw=1.4); ax.scatter([r.delta], [y + dy], marker=mk[cnd], color=col, s=18, zorder=3)
            ax.text(1.02, y + dy, f"{r.delta:+.3f} ({r.delta_lo:+.3f} to {r.delta_hi:+.3f})", va="center", fontsize=5.8, transform=ax.get_yaxis_transform())
    ax.axvline(0, color=COL["chance"], lw=0.8, ls="--"); ax.set_yticks(range(len(models))); ax.set_yticklabels([short(m) for m in models[::-1]]); ax.set_xlim(-0.25, 0.2)
    ax.set_xlabel("ΔAUROC versus full image (paired, case-clustered 95% CI)"); ax.set_title("B  Paired change (■ sector removed, ▲ sector only)", loc="left", fontsize=8)
    for j, (lab, pth) in enumerate(ex.items()):
        ax = fig.add_subplot(gs[1, j])
        if pth is None: ax.axis("off"); continue
        r = Q.loc[pth]; draw_box(ax, Image.open(pth), dict(box=(r.box_x0, r.box_y0, r.box_x1, r.box_y1), status=r.status))
        ax.set_title(("C  " if j == 0 else "") + lab + ("\nheuristically detected sonographic region" if r.status == "detected" else "\ndetection failed; fixed central box"), loc="left", fontsize=6.5)
    save_fig(fig, "Fig4_masking_control")

def fig5():
    F = load_meta(); us = F[F.modality == "ultrasound"]; src = json.loads((ROUND3 / "tables" / "MD5_roc_source.json").read_text())
    MD3 = pd.read_csv(ROUND3 / "tables" / "MD3_metadata_baseline.csv"); prim = MD3[(MD3.modality == "ultrasound") & (MD3.feature_set == PRIMARY_SET) & (MD3.model == PRIMARY_MODEL)].iloc[0]
    fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.5), gridspec_kw=dict(wspace=0.6, width_ratios=[1, 1, 1, 1.35])); rng = np.random.default_rng(SEED)
    for ax, (col, lab, letter) in zip(axes[:3], [("width", "Frame width (px)", "A"), ("aspect", "Aspect ratio", "B"), ("near_black", "Near-black fraction", "C")]):
        for i, (cl, name, c) in enumerate([(0, "benign", "#8c8c8c"), (1, "PTC", COL["us"])]):
            v = us[us.label == cl][col].values; ax.scatter(np.full(len(v), i) + rng.uniform(-0.18, 0.18, len(v)), v, s=4, alpha=0.5, color=c, lw=0)
            ax.plot([i - 0.25, i + 0.25], [np.median(v)] * 2, color="#111", lw=1.2)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["benign", "PTC"]); ax.set_ylabel(lab); ax.set_title(f"{letter}  {lab}", loc="left", fontsize=7.5)
    ax = axes[3]; s = src["ultrasound"]; fpr, tpr, _ = roc_curve(s["y"], s["p"])
    ax.plot(fpr, tpr, color=COL["us"], lw=1.6)
    ax.plot([0, 1], [0, 1], ls="--", color=COL["chance"], lw=0.8); ax.set_aspect("equal")
    ax.text(0.5, -0.42, f"frozen test AUROC {prim.test_auroc:.3f} ({prim.test_auroc_lo:.3f}–{prim.test_auroc_hi:.3f})\ngrouped CV AUROC {prim.cv_pooled_auroc:.3f} ({prim.cv_auroc_lo:.3f}–{prim.cv_auroc_hi:.3f})", transform=ax.transAxes, ha="center", va="top", fontsize=5.8)
    ax.set_xlabel("1 − specificity"); ax.set_ylabel("Sensitivity"); ax.set_title("D  Metadata-only model", loc="left", fontsize=7.5)
    save_fig(fig, "Fig5_acquisition_metadata_shortcut")

def calib_fit(y, p):
    from scipy.optimize import minimize
    z = to_logit(p); y = np.asarray(y, float)
    nll = lambda t, sl: np.sum(np.logaddexp(0, t[0] + (t[1] if sl else 1.0) * z) - y * (t[0] + (t[1] if sl else 1.0) * z))
    citl = minimize(lambda t: nll([t[0], 1.0], False), [0.0], method="BFGS").x[0]
    slope = minimize(lambda t: nll(t, True), [0.0, 1.0], method="BFGS").x[1] if np.std(z) > 0.01 else np.nan
    return citl, slope

def fig6():
    tn = pd.read_csv(ROUND2 / "predictions" / "external_TN3K_test.csv"); dd = pd.read_csv(ROUND2 / "predictions" / "external_DDTI_original.csv")
    fig = plt.figure(figsize=(7.2, 7.4)); gs = fig.add_gridspec(2, 2, hspace=0.85, wspace=0.55)
    au = {k: roc_auc_score(tn.label, tn[f"prob_{k}"]) for k in MODELS}; best = max(au, key=au.get); worst = min([k for k in MODELS if np.std(tn[f"prob_{k}"]) > 1e-3], key=au.get)
    ax = fig.add_subplot(gs[0, 0]); cur = []
    for k in ["ensemble", best, worst]:
        ci = boot_ci(tn.label, tn[f"prob_{k}"], np.arange(len(tn)))["auroc"]
        cur.append(dict(y=tn.label.values, p=tn[f"prob_{k}"].values, key=k, label=f"{DISPLAY[k].replace(' (mean of 5)', '')}: {roc_auc_score(tn.label, tn[f'prob_{k}']):.3f} ({ci[0]:.3f}–{ci[1]:.3f})"))
    roc_panel(ax, cur, "A  TN3K post-exposure reassessment\n(distributed benign/malignant labels)", legend_title="AUROC (95% CI: image-level bootstrap; no patient IDs)")
    ax = fig.add_subplot(gs[0, 1]); y, p = tn.label.values, tn.prob_ensemble.values
    b = pd.qcut(p, 10, labels=False, duplicates="drop"); g = pd.DataFrame(dict(y=y, p=p, b=b)).groupby("b").agg(n=("y", "size"), mp=("p", "mean"), ob=("y", "mean"))
    z = 1.96; lo = (g.ob + z * z / (2 * g.n) - z * np.sqrt(g.ob * (1 - g.ob) / g.n + z * z / (4 * g.n ** 2))) / (1 + z * z / g.n)
    hi = (g.ob + z * z / (2 * g.n) + z * np.sqrt(g.ob * (1 - g.ob) / g.n + z * z / (4 * g.n ** 2))) / (1 + z * z / g.n)
    ax.errorbar(g.mp, g.ob, yerr=[g.ob - lo, hi - g.ob], fmt="o", ms=3.5, color=MODEL_COL["ensemble"], elinewidth=0.8, capsize=1.5)
    ax.plot([0, 1], [0, 1], ls="--", color=COL["chance"], lw=0.8); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    citl, slope = calib_fit(y, p)
    ax.text(0.03, 0.97, f"mean predicted {p.mean():.2f} vs prevalence {y.mean():.2f}\ncalibration-in-the-large {citl:.2f}\ncalibration slope {slope:.2f}\n10 equal-count bins ({int(g.n.min())}–{int(g.n.max())} images each)", transform=ax.transAxes, va="top", fontsize=6)
    ax.set_xlabel("Mean predicted probability (decile)"); ax.set_ylabel("Observed malignant fraction"); ax.set_title("B  TN3K calibration, ensemble\n(Wilson 95% CI per bin)", loc="left", fontsize=8)
    v = dd[dd.y_primary.isin([0, 1])]; ax = fig.add_subplot(gs[1, 0]); cur = []
    for k in ["ensemble", best, worst]:
        ci = boot_ci(v.y_primary.astype(int), v[f"prob_{k}"], v.case_id.astype(str))["auroc"]
        cur.append(dict(y=v.y_primary.astype(int).values, p=v[f"prob_{k}"].values, key=k, label=f"{DISPLAY[k].replace(' (mean of 5)', '')}: {roc_auc_score(v.y_primary.astype(int), v[f'prob_{k}']):.3f} ({ci[0]:.3f}–{ci[1]:.3f})"))
    roc_panel(ax, cur, "C  DDTI concordance with TI-RADS 2–3 vs 4a–5\n(suspicion category, not malignancy)", legend_title="AUROC (95% CI: case-clustered bootstrap)")
    ax = fig.add_subplot(gs[1, 1]); items = []
    for col, lab in [("y_primary", "TI-RADS 2–3 vs 4a–5"), ("y_sens_4a_negative", "TI-RADS 2–4a vs 4b–5"), ("y_sens_exclude_3", "TI-RADS 2 vs 4–5")]:
        w = dd[dd[col].isin([0, 1])]; ci = boot_ci(w[col].astype(int), w.prob_ensemble, w.case_id.astype(str))["auroc"]
        items.append(dict(label=lab, key="ensemble", est=roc_auc_score(w[col].astype(int), w.prob_ensemble), lo=ci[0], hi=ci[1], folds=None))
    forest(ax, items, xlim=(0.3, 0.85), xlabel="AUROC, ensemble (case-clustered 95% CI)"); ax.set_title("D  DDTI: grouping sensitivity", loc="left", fontsize=8)
    save_fig(fig, "Fig6_post_exposure_reassessment")

def supp_figs():
    C2 = pd.read_csv(ROUND2 / "tables" / "C2_cv_fold_level.csv")
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), sharex=True)
    for ax, mo in zip(axes, ["cytology", "ultrasound"]):
        d = C2[C2.modality == mo]; names = list(dict.fromkeys(d.model))
        for i, m in enumerate(names[::-1]):
            v = d[d.model == m].auroc.values; ax.scatter(v, np.full(len(v), i), s=12, color=MODEL_COL[{x: y for y, x in DISPLAY.items()}[m]])
        ax.set_yticks(range(len(names))); ax.set_yticklabels([n.replace(" (mean of 5)", "") for n in names[::-1]]); ax.axvline(0.5, ls="--", color=COL["chance"], lw=0.8)
        ax.set_title(mo.capitalize(), loc="left", fontsize=8); ax.set_xlabel("Fold AUROC (grouped CV)")
    fig.tight_layout(); save_fig(fig, "S_fold_level_auroc")
    I3 = pd.read_csv(ROUND2 / "tables" / "I3_tsne_source.csv"); grp = I3.dataset.map(lambda s: "dev" if s.startswith("Pang") else s)
    fig, ax = plt.subplots(figsize=(4.6, 4.4))
    for key, lab, c in [("dev", "Development archive (Pang et al.)", COL["us"]), ("TN3K_test", "TN3K test", COL["ext"]), ("DDTI_original", "DDTI", "#6a9a6a")]:
        m = grp == key; ax.scatter(I3.tsne1[m], I3.tsne2[m], s=5, alpha=0.6, color=c, lw=0, label=f"{lab} (n = {m.sum()})")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title("Exploratory two-dimensional visualization (t-SNE);\ncluster spacing is not a measure of distance between cohorts", fontsize=7.5)
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.03), fontsize=6.5); save_fig(fig, "S_tsne_exploratory")
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
    tn = pd.read_csv(ROUND2 / "predictions" / "external_TN3K_test.csv"); dd = pd.read_csv(ROUND2 / "predictions" / "external_DDTI_original.csv"); v = dd[dd.y_primary.isin([0, 1])]
    for ax, (d, ycol, t) in zip(axes, [(tn, "label", "TN3K test, all backbones"), (v, "y_primary", "DDTI TI-RADS 2–3 vs 4a–5, all backbones")]):
        roc_panel(ax, [dict(y=d[ycol].astype(int).values, p=d[f"prob_{k}"].values, key=k, label=f"{DISPLAY[k].replace(' (mean of 5)', '')} {roc_auc_score(d[ycol].astype(int), d[f'prob_{k}']):.3f}") for k in MODELS + ["ensemble"]], t)
    save_fig(fig, "S_external_all_backbones")
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))
    for ax, mo in zip(axes, ["cytology", "ultrasound"]):
        t = pd.read_csv(EXP1B_DIR / mo / "ensemble" / "test_predictions.csv"); nb = 5
        b = pd.qcut(t.prob_malignant.rank(method="first"), nb, labels=False); g = t.assign(b=b).groupby("b").agg(n=("label", "size"), mp=("prob_malignant", "mean"), ob=("label", "mean"))
        ax.plot(g.mp, g.ob, "o-", color=COL["cyto"] if mo == "cytology" else COL["us"], ms=3.5, lw=1)
        for _, r in g.iterrows(): ax.text(r.mp, r.ob + 0.05, str(int(r.n)), fontsize=5.5, ha="center", color="#666")
        ax.plot([0, 1], [0, 1], ls="--", color=COL["chance"], lw=0.8); ax.set_aspect("equal"); ax.set_xlim(0, 1); ax.set_ylim(-0.02, 1.08)
        ax.set_title(f"{mo.capitalize()} ensemble, frozen test ({nb} equal-count bins)", loc="left", fontsize=7.5); ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed PTC fraction")
    save_fig(fig, "S_internal_calibration")
    t = pd.read_csv(EXP1B_DIR / "cytology" / "ensemble" / "test_predictions.csv"); fig, ax = plt.subplots(figsize=(3.4, 3.4))
    roc_panel(ax, [dict(y=t.label.values, p=t.prob_malignant.values, key="ensemble", label=f"Ensemble, block level {roc_auc_score(t.label, t.prob_malignant):.3f}")], "Cytology frozen test, block level")
    save_fig(fig, "S_cytology_block_level_roc")

with stage("FIG", "Figures") as go:
  if go:
    c = counts(); (ROUND3 / "tables" / "FIG_counts.json").write_text(json.dumps(c, indent=1, default=int))
    for fn in [lambda: fig1(c), fig2, fig3, fig4, fig5, fig6, supp_figs]:
        try: fn()
        except Exception as e: print("FIGURE FAILED:", getattr(fn, "__name__", "fig1"), "->", repr(e)); traceback.print_exc()

## Stage TAB — Main tables 1–4 and supplementary tables (editor points 12, 15, 17, 43–50)

In [ ]:
with stage("TAB", "Tables") as go:
  if go:
    c = json.loads((ROUND3 / "tables" / "FIG_counts.json").read_text())
    C1 = pd.read_csv(ROUND2 / "tables" / "C1_cv_summary.csv"); F2 = pd.read_csv(ROUND2 / "tables" / "F2_frozen_test_case_level.csv")
    T2 = C1[["modality", "model", "dev_cases", "dev_images", "pooled_oof_auroc", "pooled_oof_auprc", "pooled_brier"]].merge(
        F2[["modality", "model", "n", "auroc", "threshold_from_val", "tn", "fp", "fn", "tp", "sensitivity", "specificity"]].rename(columns={"n": "test_cases", "auroc": "test_case_auroc"}), on=["modality", "model"], how="left")
    save_table(T2, "T2_internal_case_level", "Primary internal case-level performance")
    rows = []
    SD = pd.read_csv(ROUND3 / "tables" / "SD1_split_design_paired.csv")
    for _, r in SD.iterrows(): rows.append(dict(experiment="split design (cytology)", model=r.model, comparison="image-level − case-grouped folds", estimate=fmt3(r.delta, r.delta_lo, r.delta_hi), p=round(r.bootstrap_p, 3), unit="ΔAUROC, paired, case-clustered"))
    MA = pd.read_csv(ROUND3 / "tables" / "MA1_masking_paired.csv")
    for _, r in MA.iterrows(): rows.append(dict(experiment="masking (ultrasound)", model=r.model, comparison=f"{r.condition} − full image", estimate=fmt3(r.delta, r.delta_lo, r.delta_hi), p=round(r.bootstrap_p, 3), unit="ΔAUROC, paired, case-clustered"))
    MD3 = pd.read_csv(ROUND3 / "tables" / "MD3_metadata_baseline.csv")
    for _, r in MD3[(MD3.feature_set == PRIMARY_SET) & (MD3.model == PRIMARY_MODEL)].iterrows():
        rows.append(dict(experiment=f"metadata-only diagnostic ({r.modality})", model="logistic regression, feature set B", comparison="frozen test", estimate=fmt3(r.test_auroc, r.test_auroc_lo, r.test_auroc_hi), p="", unit="AUROC, case-clustered"))
        rows.append(dict(experiment=f"metadata-only diagnostic ({r.modality})", model="logistic regression, feature set B", comparison="grouped CV, pooled", estimate=fmt3(r.cv_pooled_auroc, r.cv_auroc_lo, r.cv_auroc_hi), p="", unit="AUROC, case-clustered"))
    save_table(pd.DataFrame(rows), "T3_controlled_experiments", "Controlled evaluation-design experiments")
    tn = pd.read_csv(ROUND2 / "predictions" / "external_TN3K_test.csv"); tv = pd.read_csv(ROUND2 / "predictions" / "external_TN3K_trainval.csv"); dd = pd.read_csv(ROUND2 / "predictions" / "external_DDTI_original.csv")
    thr = {k: float(json.loads((EXP1B_DIR / "ultrasound" / k / "metrics.json").read_text())["image_level_threshold_from_val"]) for k in MODELS + ["ensemble"]}
    rows = []
    for nm, d in [("TN3K test", tn), ("TN3K trainval", tv)]:
        for k in MODELS + ["ensemble"]:
            y, p = d.label.values, d[f"prob_{k}"].values; m = full_metrics(y, p, thr[k]); ci = boot_ci(y, p, np.arange(len(y)), thr[k]); cit, sl = calib_fit(y, p)
            rows.append(dict(cohort=nm, endpoint="distributed benign/malignant label", unit="image (image-level bootstrap; no patient IDs)", model=DISPLAY[k], images=len(y),
                             auroc=fmt3(m["auroc"], *ci["auroc"]), auprc=round(m["auprc"], 3), no_skill=round(m["prevalence"], 3), sensitivity=round(m["sensitivity"], 3), specificity=round(m["specificity"], 3),
                             mean_predicted=round(p.mean(), 3), prevalence=round(y.mean(), 3), calibration_in_the_large=round(cit, 2), calibration_slope=("not estimable (near-constant predictions)" if np.isnan(sl) else round(sl, 2))))
    for col, lab in [("y_primary", "TI-RADS 2–3 vs 4a–5"), ("y_sens_4a_negative", "TI-RADS 2–4a vs 4b–5"), ("y_sens_exclude_3", "TI-RADS 2 vs 4–5")]:
        w = dd[dd[col].isin([0, 1])]
        for k in MODELS + ["ensemble"]:
            y, p = w[col].astype(int).values, w[f"prob_{k}"].values; ci = boot_ci(y, p, w.case_id.astype(str))
            rows.append(dict(cohort="DDTI original", endpoint=f"concordance with {lab}", unit="case-clustered bootstrap", model=DISPLAY[k], images=len(y), auroc=fmt3(roc_auc_score(y, p), *ci["auroc"]),
                             auprc=round(average_precision_score(y, p), 3), no_skill=round(y.mean(), 3)))
    save_table(pd.DataFrame(rows), "T4_post_exposure_reassessment", "Post-exposure external reassessment (TN3K) and TI-RADS concordance (DDTI)")
    fm = read_frozen(); UNCERTAIN_BEN = ["BEN_0003", "BEN_0019", "BEN_0020", "BEN_0025", "BEN_0036", "BEN_0038", "BEN_0039", "BEN_0044", "BEN_0055", "BEN_0061", "BEN_0116", "BEN_0117"]
    rows = []
    for mo in ["cytology", "ultrasound"]:
        oo = None
        for k in MODELS:
            x = pd.read_csv(CV_DIR / mo / k / "oof_predictions.csv", low_memory=False)[["image_path", "label", "component_id", "prob_malignant"]].rename(columns={"prob_malignant": k})
            oo = x if oo is None else oo.merge(x.drop(columns=["label", "component_id"]), on="image_path", validate="one_to_one")
        oo["p"] = oo[MODELS].mean(1); oo["case_key"] = oo.image_path.map(case_key_from_path)
        t = pd.read_csv(EXP1B_DIR / mo / "ensemble" / "test_source_group_predictions.csv")
        tt = pd.read_csv(EXP1B_DIR / mo / "ensemble" / "test_predictions.csv"); tt["case_key"] = tt.image_path.map(case_key_from_path)
        tg = tt.groupby("case_key").agg(label=("label", "first"), p=("prob_malignant", "mean")).reset_index()
        for scen, drop in [("as analysed", []), ("excluding provenance-uncertain benign cases", UNCERTAIN_BEN)]:
            a = oo[~oo.case_key.isin(drop)]; b = tg[~tg.case_key.isin(drop)]
            ca, cb = boot_ci(a.label, a.p, a.component_id)["auroc"], boot_ci(b.label, b.p, b.case_key)["auroc"]
            rows.append(dict(modality=mo, scenario=scen, cv_cases=a.case_key.nunique(), cv_pooled_auroc=fmt3(roc_auc_score(a.label, a.p), *ca), test_cases=len(b), test_case_auroc=fmt3(roc_auc_score(b.label, b.p), *cb)))
    save_table(pd.DataFrame(rows), "S_provenance_sensitivity", "Sensitivity analysis excluding benign cases whose provenance is uncertain (all benign cases in any duplicate group)")
    if (ROUND3 / "masking_QA_expert_review_form.csv").exists():
        f = pd.read_csv(ROUND3 / "masking_QA_expert_review_form.csv").fillna("")
        done = f[f.coverage_adequate_partial_inadequate.astype(str).str.strip() != ""]
        if len(done): save_table(done.groupby(["status", "coverage_adequate_partial_inadequate"]).size().reset_index(name="images"), "Q5_expert_review", "Expert review of sector boxes")

## Stage Z — Checksums, environment, Zenodo archive, release files and the upload zip (editor points 18, 20)

In [ ]:
with stage("Z", "Release and upload") as go:
  if go:
    rel = ROUND3 / "release_additions"; shutil.rmtree(rel, ignore_errors=True)
    def put(src, dst):
        src = Path(src)
        if src.exists(): (rel / dst).parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, rel / dst); return rel / dst
    for mo in ["ultrasound", "cytology"]:
        for k in MODELS:
            for f in ["oof_predictions.csv", "fold_metrics.csv", "cv_summary.json"]: put(CV_DIR / mo / k / f, f"predictions/grouped_cv/per_model/{mo}_{k}_{f}")
    for f in (ROUND2 / "cv_innerval").rglob("*.csv"): put(f, f"predictions/grouped_cv/inner_validation/{f.relative_to(ROUND2 / 'cv_innerval').as_posix()}")
    for k in ["convnext_small", "efficientnet_b3"]:
        for f in ["oof_predictions.csv", "fold_metrics.csv", "cv_summary.json"]: put(ROUND2 / "imagelevel_cv" / "cytology" / k / f, f"predictions/imagelevel_cv/cytology_{k}_{f}")
    for mode in ["surround_only", "center_only"]:
        for k in MODELS:
            for f in ["oof_predictions.csv", "fold_metrics.csv", "cv_summary.json"]: put(ROUND2 / "shortcut_cv" / mode / "ultrasound" / k / f, f"predictions/shortcut_cv/{mode}_{k}_{f}")
    for f in (ROUND3 / "predictions").glob("*"): put(f, f"predictions/round3/{f.name}")
    for f in (ROUND3 / "tables").glob("*"): put(f, f"results/round3/tables/{f.name}")
    for f in (ROUND3 / "figures").glob("*"): put(f, f"results/round3/figures/{f.name}")
    put(ROUND3 / "masking_QA_expert_review_form.csv", "manifests/masking_QA_expert_review_form.csv")
    try:
        (rel / "environment").mkdir(parents=True, exist_ok=True)
        (rel / "environment" / "pip_freeze.txt").write_text(subprocess.run([str(TORCH_PY), "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
        (rel / "environment" / "nvidia_smi.txt").write_text(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    except Exception as e: print("environment:", e)
    ck = [EXP1B_DIR / mo / k / "best_model.pt" for mo in ["cytology", "ultrasound"] for k in MODELS]
    ck += sorted(CV_DIR.rglob("best_model.pt")) + sorted((ROUND2 / "imagelevel_cv").rglob("best_model.pt")) + sorted((ROUND2 / "shortcut_cv").rglob("best_model.pt"))
    rows = []
    for p in ck:
        if p.exists():
            st = p.stat(); rows.append(dict(checkpoint=p.relative_to(ROOT).as_posix(), bytes=st.st_size, modified=datetime.datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M"), sha256=sha256_file(p)))
    (rel / "checkpoints").mkdir(parents=True, exist_ok=True); pd.DataFrame(rows).to_csv(rel / "checkpoints" / "checkpoint_sha256.csv", index=False)
    note_md(f"- Hashed {len(rows)} checkpoints.")
    if ZIP_CHECKPOINTS_FOR_ZENODO:
        zp = ROUND3 / "zenodo_frozen_split_checkpoints.zip"
        if not zp.exists():
            with zipfile.ZipFile(zp, "w", zipfile.ZIP_STORED) as z:
                for p in ck[:10]:
                    if p.exists(): z.write(p, p.relative_to(ROOT).as_posix())
        print("Zenodo checkpoint archive:", zp, f"{zp.stat().st_size / 1e9:.2f} GB")
    STATUS["Z"] = "done"; (ROUND3 / "logs" / "STATUS.json").write_text(json.dumps(STATUS, indent=2))
    up = ROUND3 / "REVISION_ROUND3_UPLOAD.zip"; cap = 45e6
    with zipfile.ZipFile(up, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        z.write(RESULTS_MD, "RESULTS.md"); z.write(ROUND3 / "logs" / "STATUS.json", "STATUS.json")
        files = sorted(p for p in rel.rglob("*") if p.is_file())
        files.sort(key=lambda p: (p.suffix == ".svg", p.suffix == ".png", p.stat().st_size))
        skipped = []
        for p in files:
            if sum(i.compress_size for i in z.infolist()) + p.stat().st_size * 0.6 > cap: skipped.append(p.relative_to(rel).as_posix()); continue
            z.write(p, "release_additions/" + p.relative_to(rel).as_posix())
        for f in sorted((ROUND3 / "logs").glob("*.log")):
            L = f.read_text(encoding="utf-8", errors="replace").splitlines(); z.writestr(f"logs/{f.name}", "\n".join(L[:30] + ["..."] + L[-200:] if len(L) > 240 else L))
        z.writestr("skipped_for_size.txt", "\n".join(skipped))
    print(f"\n>>> UPLOAD THIS FILE: {up}  ({up.stat().st_size / 1e6:.1f} MB); skipped for size: {len(skipped)}")
    print("stage status:", json.dumps(STATUS, indent=1))